# BERTidRAAF for AI-Driven Recruitment: Reproducible BERT–IDF Rare-Aware Adaptive Fusion Benchmark for CV Classification

This notebook provides a reproducible benchmark for evaluating **BERTidRAAF (Ours)** on two datasets:

1. **FairCVdb**
2. **Resume**

## Experimental protocol

1. Fixed execution protocol with a fixed seed.
2. Held-out test evaluation after model selection on the validation split.
3. Bootstrap confidence intervals for test accuracy and macro-F1.
4. Transparent cross-model comparison across lexical, neural, transformer, and BERT–IDF hybrid families.

## BERTidRAAF architecture

**BERTidRAAF (Ours)** is based on **BERT + IDF** and includes the following components:

1. **BERT contextual encoder** using `bert-large-uncased`.
2. **Post-contextual IDF token weighting** applied to contextual token representations.
3. **Rare-aware gated pooling** to combine contextual evidence and IDF-weighted lexical evidence.
4. **Multi-head BERT/IDF logits** using contextual, IDF, max-pooling, and interaction heads.
5. **Trainable multi-head BERT/IDF logit fusion** initialized from stable fusion weights.
6. **Multi-sample dropout regularization** over the BERT/IDF decision heads.
7. **Validation-selected BERT–IDF expert fusion** selected only on the validation split.
8. **BERT-anchored lexical evidence** through TF-IDF/IDF branches used as complementary evidence.

## Benchmark families

1. Lexical baselines: TF-IDF + SVM, Logistic Regression, and Naive Bayes.
2. Neural baselines: CNN, LSTM, and BiLSTM.
3. Transformer baselines: BERTBASE, BERTLARGE, RoBERTaBASE, and DeBERTaBASE.
4. BERT–IDF hybrid baselines: BERT+IDF, BERT+CNN, BERT–IDF+CNN, BERT–IDF+LSTM, and BERT–IDF+BiLSTM.


## 1. Imports and global configuration


In [ ]:
# Install in a fresh Colab runtime if dependencies are missing:
# !pip -q install transformers accelerate openpyxl scipy scikit-learn matplotlib pandas numpy tqdm

import gc, html, inspect, json, math, os, random, re, time, warnings, zipfile
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

from scipy.sparse import hstack
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, auc, average_precision_score, classification_report,
                             confusion_matrix, f1_score, precision_recall_curve, precision_recall_fscore_support,
                             roc_auc_score, roc_curve)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import label_binarize, normalize
from sklearn.svm import LinearSVC

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
is_h100 = "H100" in gpu_name.upper()
print("Device:", device, "| GPU:", gpu_name)

CONFIG = {
    "proposed_model_name": "BERTidRAAF (Ours)",
    "proposed_model_long_name": "BERT with IDF-aware Rare-Aware Adaptive Fusion",
    "seed": 42,
    "datasets_to_run": ["FairCVdb", "Resume"],

    "test_size": 0.20,
    "validation_size_inside_train": 0.10,

    "run_classical_models": True,
    "run_recurrent_models": True,
    "run_convolutional_models": True,
    "run_transformer_models": True,

    "max_length": 384,
    "proposed_max_length": 512,
    "epochs_proposed": 8,
    "epochs_transformer": 5,
    "epochs_rnn": 8,
    "epochs_cnn": 8,
    "batch_size_base": 8 if is_h100 else 2,
    "batch_size_large": 4 if is_h100 else 1,
    "grad_accum_base": 2 if is_h100 else 8,
    "grad_accum_large": 4 if is_h100 else 16,
    "early_stopping_patience": 2,

    "lr": 8e-6,
    "head_lr": 3e-5,
    "weight_decay": 1e-2,
    "warmup_ratio": 0.08,
    "dropout": 0.10,
    "raaf_msd_dropout_probs": [0.05, 0.10, 0.15, 0.20],
    "raaf_trainable_head_mixing": True,
    "label_smoothing": 0.02,
    "idf_alpha": 1.60,
    "gradient_checkpointing": True,

    "tfidf_word_max_features": 180000,
    "tfidf_char_max_features": 180000,
    "fusion_weight_grid_step": 0.05,
    "fusion_temperature_grid": [0.60, 0.70, 0.85, 1.00, 1.20, 1.50],
    "allow_geometric_fusion": True,

    "dataset_model_config": {
        "Resume": {
            "backbone": "bert-large-uncased",
            "max_length": 512,
            "epochs": 8,
            "proposed_seeds": [42],
            "candidate_policy": "core_preservation",
            "min_w_bert": 0.25,
            "min_w_logreg": 0.05,
            "min_w_svm": 0.00,
            "min_w_centroid": 0.00,
            "min_w_nb": 0.00,
            "min_class_count": 5,
        },
        "FairCVdb": {
            "backbone": "bert-large-uncased",
            "max_length": 512,
            "epochs": 8,
            "top_k_checkpoints": 1,
            "patience": 2,
            "lr": 8e-6,
            "head_lr": 3e-5,
            "proposed_seeds": [42],
            "candidate_policy": "bertidf_validation_selection",
            "bert_anchor_grid": [0.01, 0.02, 0.03, 0.05, 0.08, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50],
            "confidence_tau_grid": [0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90],
            "confidence_alpha_grid": [0.20, 0.35, 0.50, 0.65],
            "threshold_grid": [round(x, 2) for x in np.arange(0.25, 0.76, 0.01)],
            "override_tau_grid": [round(x, 2) for x in np.arange(0.55, 0.96, 0.05)],
            "override_margin_grid": [round(x, 2) for x in np.arange(-0.10, 0.31, 0.05)],
            "min_w_bert": 0.05,
            "min_w_logreg": 0.05,
            "min_w_svm": 0.00,
            "min_w_centroid": 0.00,
            "min_w_nb": 0.00,
            "min_class_count": 1,
        },
    },

    "output_dir": "/content/bertidraaf_stronger_bert_idf_msd_outputs",
}
Path(CONFIG["output_dir"]).mkdir(parents=True, exist_ok=True)

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
seed_everything(CONFIG["seed"])


## 2. Utilities: downloading, preprocessing, metrics, and IDF weighting


In [ ]:
DATA_URLS = {
    "Resume": "https://raw.githubusercontent.com/omarhaddad-projects/BERTidRAAF_for_AI-Driven_Recruitment/main/Data/Resume.zip",
    "FairCVdb": "https://raw.githubusercontent.com/omarhaddad-projects/BERTidRAAF_for_AI-Driven_Recruitment/main/Data/FairCVdb.zip"
}
DATA_DIR = Path("/content/data"); DATA_DIR.mkdir(parents=True, exist_ok=True)

def cleanup():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def github_to_raw(url):
    return url.replace("https://github.com/", "https://raw.githubusercontent.com/").replace("/blob/", "/")

def github_to_raw_url(url):
    """
    Convert GitHub browser links to raw file links when needed.
    """
    url = str(url)
    if "github.com" in url and "/blob/" in url:
        return url.replace("https://github.com/", "https://raw.githubusercontent.com/").replace("/blob/", "/")
    if "github.com" in url and "/tree/" in url:
        return url.replace("https://github.com/", "https://raw.githubusercontent.com/").replace("/tree/", "/")
    return url

def download_file(url, out_path):
    url = github_to_raw_url(url)
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    if out_path.exists() and out_path.stat().st_size > 0:
        print("Already downloaded:", out_path)
        return out_path

    print("Downloading:", url)
    response = requests.get(url, stream=True, timeout=120)
    response.raise_for_status()

    with open(out_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

    return out_path


def normalize_text(x):
    x = "" if pd.isna(x) else str(x)
    x = html.unescape(x); x = re.sub(r"<[^>]+>", " ", x); x = re.sub(r"\s+", " ", x)
    return x.strip()

def canonicalize_label(x):
    x = "" if pd.isna(x) else str(x)
    x = html.unescape(x); x = re.sub(r"\s+", " ", x).strip()
    return x.upper()

def tokenize_simple(text):
    return re.findall(r"[a-zA-Z][a-zA-Z0-9\+\#\.-]+", str(text).lower())

def build_idf_dictionary(texts):
    docs = [set(tokenize_simple(t)) for t in texts]
    n = max(len(docs), 1); df = Counter()
    for d in docs: df.update(d)
    idf = {t: math.log((1+n)/(1+c)) + 1 for t,c in df.items()}
    vals = list(idf.values()) if idf else [1.0]
    return idf, float(math.log((1+n)/1)+1), float(min(vals)), float(max(vals))

def safe_proba(scores):
    s = np.asarray(scores, dtype=float)
    if s.ndim == 1: s = np.vstack([-s, s]).T
    s = s - s.max(axis=1, keepdims=True)
    e = np.exp(s); return e / e.sum(axis=1, keepdims=True).clip(min=1e-12)

def multiclass_auc(y_true, proba, n_classes):
    try:
        if proba is None: return np.nan
        if n_classes == 2 and proba.ndim == 2 and proba.shape[1] == 2:
            return roc_auc_score(y_true, proba[:,1])
        yb = label_binarize(y_true, classes=np.arange(n_classes))
        return roc_auc_score(yb, proba, average="macro", multi_class="ovr")
    except Exception:
        return np.nan

def compute_metrics(y_true, y_pred, proba, label_order):
    pr, rc, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    wpr, wrc, wf1, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_precision": pr, "macro_recall": rc, "macro_f1": f1,
        "weighted_precision": wpr, "weighted_recall": wrc, "weighted_f1": wf1,
        "micro_f1": f1_score(y_true, y_pred, average="micro"),
        "auc_macro": multiclass_auc(np.asarray(y_true), proba, len(label_order)),
        "report": classification_report(y_true, y_pred, labels=np.arange(len(label_order)),
                                        target_names=label_order, output_dict=True, zero_division=0)
}

def bootstrap_metric_ci(y_true, y_pred, metric_fn, n_boot=500, seed=42):
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    n = len(y_true)

    if n == 0:
        return np.nan, np.nan, np.nan

    vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        vals.append(metric_fn(y_true[idx], y_pred[idx]))

    return float(np.mean(vals)), float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def bootstrap_accuracy_ci(y_true, y_pred, n_boot=500, seed=42):
    return bootstrap_metric_ci(
        y_true,
        y_pred,
        lambda yt, yp: accuracy_score(yt, yp),
        n_boot=n_boot,
        seed=seed,
    )

def bootstrap_macro_f1_ci(y_true, y_pred, n_boot=500, seed=42):
    return bootstrap_metric_ci(
        y_true,
        y_pred,
        lambda yt, yp: f1_score(yt, yp, average="macro", zero_division=0),
        n_boot=n_boot,
        seed=seed,
    )

def count_parameters(model):
    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))


## 3. Dataset loaders: Resume and FairCVdb


In [ ]:
def load_resume_dataset():
    z = download_file(DATA_URLS["Resume"], DATA_DIR/"Resume.zip")
    extract = DATA_DIR/"Resume"; extract.mkdir(exist_ok=True)
    with zipfile.ZipFile(z, "r") as f: f.extractall(extract)
    csvs = list(extract.rglob("*.csv"))
    if not csvs: raise FileNotFoundError("No CSV in Resume.zip")
    df = pd.read_csv(csvs[0])
    text_col = "Resume_str" if "Resume_str" in df.columns else next(c for c in df.columns if "resume" in c.lower() or "text" in c.lower())
    label_col = "Category" if "Category" in df.columns else next(c for c in df.columns if "category" in c.lower() or "label" in c.lower())
    out = pd.DataFrame({"text": df[text_col].map(normalize_text), "label": df[label_col].map(canonicalize_label), "dataset":"Resume"})
    return out[out.text.str.len()>20].drop_duplicates(["text","label"]).reset_index(drop=True)

def stringify_value(v):
    if v is None: return ""
    if isinstance(v, bytes): return v.decode("utf-8", errors="ignore")
    if isinstance(v, dict): return " ".join(f"{k}: {stringify_value(x)}" for k,x in v.items())
    if isinstance(v, (list, tuple, np.ndarray)):
        arr=np.asarray(v, dtype=object).reshape(-1)
        return " ".join(stringify_value(x) for x in arr if stringify_value(x).strip())
    s=str(v); return "" if s.lower() in {"nan","none"} else s

def array_to_text_list(obj):
    arr=np.asarray(obj, dtype=object)
    if arr.shape == (): arr=np.asarray(arr.item(), dtype=object)
    if arr.ndim <= 1: return [stringify_value(x) for x in arr.reshape(-1).tolist()]
    return [stringify_value(row) for row in arr]

def labels_to_score_list(obj):
    arr=np.asarray(obj, dtype=object)
    if arr.shape == (): arr=np.asarray(arr.item(), dtype=object)
    if arr.ndim == 1:
        vals=[]
        for x in arr.tolist():
            if isinstance(x,(list,tuple,np.ndarray)): vals.append(float(np.asarray(x, dtype=float).reshape(-1).mean()))
            else: vals.append(float(x))
        return vals
    arr=arr.astype(float)
    return arr[:,0].tolist() if arr.ndim==2 and arr.shape[1]==1 else arr.mean(axis=1).tolist()

def load_faircvdb_dataset():
    z = download_file(DATA_URLS["FairCVdb"], DATA_DIR/"FairCVdb.zip")
    extract = DATA_DIR/"FairCVdb"; extract.mkdir(exist_ok=True)
    with zipfile.ZipFile(z, "r") as f: f.extractall(extract)
    files = list(extract.rglob("*.npy")) + list(extract.rglob("*.npz"))
    if not files: raise FileNotFoundError("No FairCVdb.npy/npz found")
    loaded=np.load(files[0], allow_pickle=True)
    data={k:loaded[k] for k in loaded.keys()} if isinstance(loaded, np.lib.npyio.NpzFile) else loaded.item()
    required=["Profiles Train","Profiles Test","Bios Train","Bios Test","Blind Labels Train","Blind Labels Test"]
    missing=[k for k in required if k not in data]
    if missing: raise KeyError(f"Missing FairCVdb keys: {missing}")
    pt,bt,st = array_to_text_list(data["Profiles Train"]), array_to_text_list(data["Bios Train"]), labels_to_score_list(data["Blind Labels Train"])
    pe,be,se = array_to_text_list(data["Profiles Test"]), array_to_text_list(data["Bios Test"]), labels_to_score_list(data["Blind Labels Test"])
    nt,ne=min(len(pt),len(bt),len(st)),min(len(pe),len(be),len(se))
    threshold=float(np.median(st[:nt])); print("FairCVdb train-median binary threshold:", threshold)
    def combine(a,b): return (str(a).strip()+". "+str(b).strip()).strip(". ")
    def lab(x): return "HIGH-SUITABILITY" if float(x)>=threshold else "LOW-SUITABILITY"
    train=pd.DataFrame({"text":[normalize_text(combine(pt[i],bt[i])) for i in range(nt)],"score":st[:nt],"label":[lab(x) for x in st[:nt]],"dataset":"FairCVdb","source_split":"train"})
    test=pd.DataFrame({"text":[normalize_text(combine(pe[i],be[i])) for i in range(ne)],"score":se[:ne],"label":[lab(x) for x in se[:ne]],"dataset":"FairCVdb","source_split":"test"})
    for key,col,split in [("Biased Labels Train (Gender)","gender",train),("Biased Labels Test (Gender)","gender",test),("Biased Labels Train (Ethnicity)","ethnicity",train),("Biased Labels Test (Ethnicity)","ethnicity",test)]:
        if key in data:
            vals=array_to_text_list(data[key])
            if len(vals)>=len(split): split[col]=vals[:len(split)]
    train=train[train.text.str.len()>20].drop_duplicates(["text","label"]).reset_index(drop=True)
    test=test[test.text.str.len()>20].drop_duplicates(["text","label"]).reset_index(drop=True)
    return train, test, {"faircvdb_binary_threshold": threshold, "has_official_split": True}

def infer_linkedin_columns(df):
    cols=list(df.columns); low={c:str(c).lower() for c in cols}
    text_cols=[c for c in cols if any(k in low[c] for k in ["summary","description","about","bio","profile","headline","experience","skills","text","content"])]
    lab_cols=[c for c in cols if any(k in low[c] for k in ["category","industry","job","role","title","label","class","position"]) and not any(b in low[c] for b in ["url","link","name","id"])]
    if not text_cols: text_cols=[c for c in cols if df[c].dtype=="object"][:3]
    if not lab_cols:
        for c in cols:
            if c not in text_cols and 2 <= df[c].nunique(dropna=True) <= max(200, len(df)//10): lab_cols=[c]; break
    if not text_cols or not lab_cols: raise ValueError(f"Could not infer columns: {cols}")
    return text_cols, lab_cols[0]

def load_linkedin_dataset():
    p=download_file(DATA_URLS["LinkedIn"], DATA_DIR/"Dump.csv")
    df=pd.read_csv(p)
    text_cols,label_col=infer_linkedin_columns(df)
    print("LinkedIn text columns:", text_cols, "| label:", label_col)
    text=df[text_cols].fillna("").astype(str).agg(" ".join, axis=1)
    out=pd.DataFrame({"text":text.map(normalize_text),"label":df[label_col].map(canonicalize_label),"dataset":"LinkedIn"})
    return out[(out.text.str.len()>20) & (out.label.str.len()>0)].drop_duplicates(["text","label"]).reset_index(drop=True)

def load_dataset_by_name(name):
    if name=="Resume": return load_resume_dataset(), None, {}
    if name=="FairCVdb": return load_faircvdb_dataset()
    if name=="LinkedIn": return load_linkedin_dataset(), None, {}
    raise ValueError(name)


## 4. Build dataset bundles and train/validation/test splits


In [ ]:
def filter_rare_classes_transparently(df, dataset_name):
    """
    Filter extremely rare classes before splitting.
    This is mainly important for LinkedIn, where some automatically inferred
    labels may have too few examples for a meaningful train/validation/test split.

    The operation is transparent: the number of removed rows/classes is stored
    in the dataset metadata and exported in the accounting tables.
    """
    cfg = CONFIG["dataset_model_config"].get(dataset_name, {})
    min_count = int(cfg.get("min_class_count", 1))

    if min_count <= 1:
        return df.reset_index(drop=True), {
            "min_class_count": min_count,
            "removed_rows_by_min_class_filter": 0,
            "removed_classes_by_min_class_filter": 0,
        }

    counts = df["label"].value_counts()
    keep_labels = set(counts[counts >= min_count].index.tolist())
    removed_classes = int((counts < min_count).sum())
    before = len(df)
    filtered = df[df["label"].isin(keep_labels)].copy().reset_index(drop=True)

    meta = {
        "min_class_count": min_count,
        "removed_rows_by_min_class_filter": int(before - len(filtered)),
        "removed_classes_by_min_class_filter": removed_classes,
    }

    if before != len(filtered):
        print(
            f"[{dataset_name}] Rare-class filter: kept {len(filtered)}/{before} rows; "
            f"removed {removed_classes} classes with support < {min_count}."
        )

    return filtered, meta

def split_standard(df, name):
    df, filter_meta = filter_rare_classes_transparently(df, name)
    df = df.copy().reset_index(drop=True)

    labels = sorted(df.label.unique().tolist())
    lid = {l:i for i,l in enumerate(labels)}
    df["label_id"] = df.label.map(lid).astype(int)

    cnt = df.label_id.value_counts()
    ncls = len(cnt)
    ntest = max(1, int(round(CONFIG["test_size"]*len(df))))

    strat = df.label_id if cnt.min()>=2 and ntest>=ncls and (len(df)-ntest)>=ncls else None
    if strat is None:
        print(f"[{name}] Warning: test split is non-stratified because of rare classes or class count.")

    trv, te = train_test_split(
        df,
        test_size=CONFIG["test_size"],
        random_state=CONFIG["seed"],
        stratify=strat
    )

    cnt2 = trv.label_id.value_counts()
    nval = max(1, int(round(CONFIG["validation_size_inside_train"]*len(trv))))
    strat2 = trv.label_id if cnt2.min()>=2 and nval>=ncls and (len(trv)-nval)>=ncls else None
    if strat2 is None:
        print(f"[{name}] Warning: validation split is non-stratified because of rare classes or class count.")

    tr, va = train_test_split(
        trv,
        test_size=CONFIG["validation_size_inside_train"],
        random_state=CONFIG["seed"],
        stratify=strat2
    )

    return tr.reset_index(drop=True), va.reset_index(drop=True), te.reset_index(drop=True), labels, filter_meta

def split_faircvdb(train_df, test_df):
    labels = ["HIGH-SUITABILITY","LOW-SUITABILITY"] if set(pd.concat([train_df.label,test_df.label]))=={"HIGH-SUITABILITY","LOW-SUITABILITY"} else sorted(pd.concat([train_df.label,test_df.label]).unique().tolist())
    lid = {l:i for i,l in enumerate(labels)}
    train_df = train_df.copy()
    test_df = test_df.copy()
    train_df["label_id"] = train_df.label.map(lid).astype(int)
    test_df["label_id"] = test_df.label.map(lid).astype(int)

    tr, va = train_test_split(
        train_df,
        test_size=CONFIG["validation_size_inside_train"],
        random_state=CONFIG["seed"],
        stratify=train_df.label_id
    )
    return tr.reset_index(drop=True), va.reset_index(drop=True), test_df.reset_index(drop=True), labels, {
        "min_class_count": 1,
        "removed_rows_by_min_class_filter": 0,
        "removed_classes_by_min_class_filter": 0,
    }

def build_bundle(name):
    df, official_test, meta = load_dataset_by_name(name)

    if name=="FairCVdb" and official_test is not None:
        tr, va, te, labels, filter_meta = split_faircvdb(df, official_test)
    else:
        tr, va, te, labels, filter_meta = split_standard(df, name)

    idf, default, mn, mx = build_idf_dictionary(tr.text)

    final_meta = {}
    final_meta.update(meta or {})
    final_meta.update(filter_meta or {})

    bundle = {
        "dataset": name,
        "train": tr,
        "valid": va,
        "test": te,
        "label_order": labels,
        "n_classes": len(labels),
        "idf_dict": idf,
        "default_idf": default,
        "idf_min": mn,
        "idf_max": mx,
        "meta": final_meta,
    }

    print(f"\n[{name}] Train={len(tr)} | Valid={len(va)} | Test={len(te)} | Classes={len(labels)}")
    if final_meta:
        print(f"[{name}] Meta:", final_meta)

    return bundle

DATASETS = {name: build_bundle(name) for name in CONFIG["datasets_to_run"]}

rows = []
dist = []
for n,b in DATASETS.items():
    rows += [
        {"Dataset":n,"Subset":"Train","Count":len(b["train"])},
        {"Dataset":n,"Subset":"Validation","Count":len(b["valid"])},
        {"Dataset":n,"Subset":"Test","Count":len(b["test"])},
        {"Dataset":n,"Subset":"Classes","Count":b["n_classes"]},
        {"Dataset":n,"Subset":"Min class count used","Count":b["meta"].get("min_class_count", 1)},
        {"Dataset":n,"Subset":"Rows removed by rare-class filter","Count":b["meta"].get("removed_rows_by_min_class_filter", 0)},
        {"Dataset":n,"Subset":"Classes removed by rare-class filter","Count":b["meta"].get("removed_classes_by_min_class_filter", 0)},
    ]
    for cls,cnt in b["test"].label.value_counts().sort_index().items():
        dist.append({"Dataset":n,"Class":cls,"Test Support":int(cnt)})

T01_dataset_accounting = pd.DataFrame(rows)
T02_label_distribution = pd.DataFrame(dist)
display(T01_dataset_accounting)
display(T02_label_distribution.head(30))


## 5. Classical, recurrent, and convolutional baselines


In [ ]:
CLASSICAL_MODELS = ["SVM (TF-IDF)", "LogReg (TF-IDF)", "MultinomialNB"]
RECURRENT_MODELS = ["LSTM", "BiLSTM"]
CONVOLUTIONAL_MODELS = ["CNN"]

def tfidf_matrices(bundle):
    tr, va, te = bundle["train"], bundle["valid"], bundle["test"]
    w = TfidfVectorizer(
        ngram_range=(1, 3),
        min_df=1,
        max_df=0.98,
        max_features=CONFIG["tfidf_word_max_features"],
        sublinear_tf=True,
        strip_accents="unicode"
    )
    c = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=1,
        max_features=CONFIG["tfidf_char_max_features"] // 2,
        sublinear_tf=True
    )
    Xtr = hstack([w.fit_transform(tr.text), c.fit_transform(tr.text)]).tocsr()
    Xva = hstack([w.transform(va.text), c.transform(va.text)]).tocsr()
    Xte = hstack([w.transform(te.text), c.transform(te.text)]).tocsr()
    return Xtr, Xva, Xte, tr.label_id.to_numpy(), va.label_id.to_numpy(), te.label_id.to_numpy()

def run_classical(bundle, model_name):
    Xtr, Xva, Xte, ytr, yva, yte = tfidf_matrices(bundle)

    if model_name == "SVM (TF-IDF)":
        clf = CalibratedClassifierCV(LinearSVC(), cv=3)
    elif model_name == "LogReg (TF-IDF)":
        clf = LogisticRegression(C=4.0, max_iter=3000, n_jobs=-1)
    elif model_name == "MultinomialNB":
        clf = MultinomialNB(alpha=0.05)
    else:
        raise ValueError(model_name)

    t0 = time.perf_counter()
    clf.fit(Xtr, ytr)
    train_time = time.perf_counter() - t0

    t1 = time.perf_counter()
    yp = clf.predict(Xte)
    infer = time.perf_counter() - t1

    if hasattr(clf, "predict_proba"):
        P = clf.predict_proba(Xte)
    elif hasattr(clf, "decision_function"):
        P = safe_proba(clf.decision_function(Xte))
    else:
        P = None

    m = compute_metrics(yte, yp, P, bundle["label_order"])
    return {
        "dataset": bundle["dataset"],
        "model": model_name,
        "family": "Lexical Baseline",
        "y_true": yte,
        "y_pred": yp,
        "probas": P,
        "history": [],
        "train_time_epoch_s": train_time,
        "inference_total_s": infer,
        "inference_ms_per_doc": infer / max(len(yte), 1) * 1000,
        "params": np.nan,
        **m,
    }

def build_vocab(texts, max_vocab=70000):
    cnt = Counter()
    for t in texts:
        cnt.update(tokenize_simple(t))
    vocab = {"<PAD>": 0, "<UNK>": 1}
    for tok, _ in cnt.most_common(max_vocab - 2):
        vocab[tok] = len(vocab)
    return vocab

def enc_text(text, vocab, max_len=384):
    ids = [vocab.get(t, 1) for t in tokenize_simple(text)[:max_len]]
    return ids + [0] * (max_len - len(ids))

class RNNDataset(Dataset):
    def __init__(self, df, vocab, max_len=384):
        self.x = [enc_text(t, vocab, max_len) for t in df.text]
        self.y = df.label_id.astype(int).tolist()
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return {
            "input_ids": torch.tensor(self.x[i], dtype=torch.long),
            "labels": torch.tensor(self.y[i], dtype=torch.long),
        }

class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, n_classes, bidir=False):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, 256, padding_idx=0)
        self.rnn = nn.LSTM(256, 256, batch_first=True, bidirectional=bidir)
        self.drop = nn.Dropout(0.25)
        self.fc = nn.Linear(512 if bidir else 256, n_classes)
    def forward(self, input_ids, labels=None):
        out, _ = self.rnn(self.emb(input_ids))
        feat = out.max(dim=1).values
        logits = self.fc(self.drop(feat))
        loss = F.cross_entropy(logits, labels) if labels is not None else None
        return {"loss": loss, "logits": logits}

class TextCNNClassifier(nn.Module):
    """
    CNN-only baseline. This is intentionally independent of BERT to provide
    a missing convolutional baseline comparable to LSTM and BiLSTM.
    """
    def __init__(self, vocab_size, n_classes, embed_dim=256, filters=256, kernels=(3, 4, 5), dropout=0.30):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.convs = nn.ModuleList([nn.Conv1d(embed_dim, filters, k) for k in kernels])
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(filters * len(kernels), n_classes)
    def forward(self, input_ids, labels=None):
        x = self.emb(input_ids).transpose(1, 2)
        feats = []
        for conv in self.convs:
            c = F.gelu(conv(x))
            feats.append(F.max_pool1d(c, kernel_size=c.size(2)).squeeze(2))
        feat = torch.cat(feats, dim=1)
        logits = self.fc(self.drop(feat))
        loss = F.cross_entropy(logits, labels) if labels is not None else None
        return {"loss": loss, "logits": logits}

def eval_neural_baseline(model, loader, bundle):
    model.eval()
    ys, ps, prob, losses = [], [], [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            out = model(**batch)
            losses.append(float(out["loss"].detach().cpu()))
            p = torch.softmax(out["logits"], dim=1).detach().cpu().numpy()
            prob.append(p)
            ps += p.argmax(axis=1).tolist()
            ys += batch["labels"].detach().cpu().numpy().tolist()
    prob = np.concatenate(prob)
    ys = np.array(ys)
    ps = np.array(ps)
    m = compute_metrics(ys, ps, prob, bundle["label_order"])
    m.update({"y_true": ys, "y_pred": ps, "probas": prob, "loss": float(np.mean(losses))})
    return m

def train_sequence_baseline(bundle, model_name, model_factory, family, epochs_key):
    seed_everything(CONFIG["seed"])
    vocab = build_vocab(bundle["train"].text)

    model = model_factory(len(vocab), bundle["n_classes"]).to(device)
    tr = DataLoader(RNNDataset(bundle["train"], vocab), batch_size=64, shuffle=True)
    va = DataLoader(RNNDataset(bundle["valid"], vocab), batch_size=128)
    te = DataLoader(RNNDataset(bundle["test"], vocab), batch_size=128)

    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    best, best_f1, stale = None, -1, 0
    hist, times = [], []

    for ep in range(1, CONFIG[epochs_key] + 1):
        model.train()
        t0 = time.perf_counter()
        losses, ty, tp = [], [], []

        for batch in tr:
            batch = {k: v.to(device) for k, v in batch.items()}
            opt.zero_grad(set_to_none=True)
            out = model(**batch)
            out["loss"].backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            losses.append(float(out["loss"].detach().cpu()))
            tp += out["logits"].detach().argmax(1).cpu().numpy().tolist()
            ty += batch["labels"].detach().cpu().numpy().tolist()

        times.append(time.perf_counter() - t0)
        val = eval_neural_baseline(model, va, bundle)
        hist.append({
            "epoch": ep,
            "train_loss": float(np.mean(losses)),
            "val_loss": val["loss"],
            "train_acc": accuracy_score(ty, tp),
            "val_acc": val["accuracy"],
            "train_macro_f1": f1_score(ty, tp, average="macro"),
            "val_macro_f1": val["macro_f1"],
        })

        print(f"[{bundle['dataset']}] {model_name} epoch {ep}: val_acc={val['accuracy']*100:.2f}, val_f1={val['macro_f1']:.4f}")

        if val["macro_f1"] > best_f1:
            best_f1 = val["macro_f1"]
            best = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stale = 0
        else:
            stale += 1

        if stale >= CONFIG["early_stopping_patience"]:
            break

    model.load_state_dict(best)
    t = time.perf_counter()
    test = eval_neural_baseline(model, te, bundle)
    infer = time.perf_counter() - t

    return {
        "dataset": bundle["dataset"],
        "model": model_name,
        "family": family,
        "history": hist,
        "train_time_epoch_s": float(np.mean(times)),
        "inference_total_s": infer,
        "inference_ms_per_doc": infer / max(len(bundle["test"]), 1) * 1000,
        "params": sum(p.numel() for p in model.parameters() if p.requires_grad),
        **test,
    }

def run_recurrent(bundle, model_name):
    bidir = model_name == "BiLSTM"
    return train_sequence_baseline(
        bundle,
        model_name,
        lambda vocab_size, n_classes: RNNClassifier(vocab_size, n_classes, bidir=bidir),
        "Neural Baseline",
        "epochs_rnn",
    )

def run_convolutional(bundle, model_name="CNN"):
    return train_sequence_baseline(
        bundle,
        model_name,
        lambda vocab_size, n_classes: TextCNNClassifier(vocab_size, n_classes),
        "Neural Baseline",
        "epochs_cnn",
    )


## 6. Transformer, hybrid, and BERTidRAAF architectures


In [ ]:
def expand_token(text, start, end):
    s = str(text).lower()
    n = len(s)
    while start > 0 and re.match(r"[a-z0-9\+\#\.-]", s[start - 1]):
        start -= 1
    while end < n and re.match(r"[a-z0-9\+\#\.-]", s[end]):
        end += 1
    return re.sub(r"[^a-z0-9\+\#\.-]+", "", s[start:end])

def load_tokenizer_robust(backbone):
    """
    Fixes tokenizer issues observed with some models such as DeBERTa.
    Uses a fast tokenizer when possible, then falls back to a slow tokenizer.
    """
    try:
        return AutoTokenizer.from_pretrained(backbone, use_fast=True)
    except Exception as exc:
        print(f"Fast tokenizer failed for {backbone}: {exc}")
        print("Retrying with use_fast=False.")
        return AutoTokenizer.from_pretrained(backbone, use_fast=False)

class TrDataset(Dataset):
    def __init__(self, df, tokenizer, bundle, max_len, use_idf):
        self.df = df.reset_index(drop=True)
        self.tok = tokenizer
        self.b = bundle
        self.max_len = max_len
        self.use_idf = use_idf
        self.has_tt = "token_type_ids" in tokenizer.model_input_names

    def __len__(self):
        return len(self.df)

    def idfw(self, token):
        raw = float(self.b["idf_dict"].get(token, self.b["default_idf"]))
        denom = max(self.b["idf_max"] - self.b["idf_min"], 1e-6)
        return float(np.clip(1 + CONFIG["idf_alpha"] * ((raw - self.b["idf_min"]) / denom), 1, 2.2))

    def weights(self, text, offs):
        w = []
        for st, en in offs:
            if en <= st:
                w.append(0.0)
            else:
                w.append(self.idfw(expand_token(text, int(st), int(en))))
        return (w + [0.0] * self.max_len)[:self.max_len]

    def __getitem__(self, i):
        r = self.df.iloc[i]
        text = str(r.text)

        try:
            enc = self.tok(
                text,
                truncation=True,
                padding="max_length",
                max_length=self.max_len,
                return_offsets_mapping=self.use_idf,
            )
        except NotImplementedError:
            enc = self.tok(
                text,
                truncation=True,
                padding="max_length",
                max_length=self.max_len,
                return_offsets_mapping=False,
            )

        item = {
            "input_ids": torch.tensor(enc["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(enc["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(int(r.label_id), dtype=torch.long),
        }

        if self.has_tt and "token_type_ids" in enc:
            item["token_type_ids"] = torch.tensor(enc["token_type_ids"], dtype=torch.long)

        if self.use_idf:
            if "offset_mapping" in enc:
                idf_weights = self.weights(text, enc["offset_mapping"])
            else:
                # Slow-tokenizer fallback: keep BERT running and use attention mask as neutral IDF.
                idf_weights = [float(x) for x in enc["attention_mask"]]
            item["idf_weights"] = torch.tensor(idf_weights, dtype=torch.float32)

        return item

def collate(batch):
    return {k: torch.stack([x[k] for x in batch]) for k in batch[0]}

def hsize(model):
    return int(getattr(model.config, "hidden_size", getattr(model.config, "d_model", 768)))

def inverse_softplus(values):
    values = torch.as_tensor(values, dtype=torch.float32)
    return torch.log(torch.expm1(values).clamp(min=1e-6))

class TransformerClassifier(nn.Module):
    """
    General transformer classifier used for baselines and the proposed model.

    Proposed mode: 'bertidraaf'
    - BERT contextual encoder
    - IDF-weighted pooling
    - rare-aware gated pooling
    - multi-head BERT/IDF logits
    """
    def __init__(self, backbone, n_classes, mode="mean", use_idf=False, kernels=(2, 3, 4, 5, 6), filters=256):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(backbone)
        self.accepts_token_type_ids = "token_type_ids" in inspect.signature(self.encoder.forward).parameters
        self.mode = mode
        self.use_idf = use_idf
        h = hsize(self.encoder)
        self.h = h

        if CONFIG["gradient_checkpointing"] and ("deberta" not in backbone.lower()) and hasattr(self.encoder, "gradient_checkpointing_enable"):
            self.encoder.gradient_checkpointing_enable()

        if mode == "idf_softmax":
            self.attn = nn.Linear(h, 1)

        if mode == "layer_scale":
            self.scale = nn.Parameter(torch.ones(h))

        if mode == "cnn":
            self.convs = nn.ModuleList([nn.Conv1d(h, filters, k) for k in kernels])
            outdim = filters * len(kernels) + 2 * h

        elif mode in {"lstm", "bilstm"}:
            # BERT-IDF + LSTM/BiLSTM hybrid baseline.
            # This is a close comparator to BERT-IDF+CNN and to the proposed fusion design.
            self.seq = nn.LSTM(
                h,
                h if mode == "lstm" else h // 2,
                batch_first=True,
                bidirectional=(mode == "bilstm"),
            )
            outdim = 3 * h

        elif mode == "raif":
            self.gate = nn.Sequential(
                nn.Linear(4 * h, h),
                nn.LayerNorm(h),
                nn.GELU(),
                nn.Dropout(CONFIG["dropout"]),
                nn.Linear(h, h),
            )
            self.proj = nn.Sequential(
                nn.Linear(6 * h, h),
                nn.LayerNorm(h),
                nn.GELU(),
                nn.Dropout(CONFIG["dropout"]),
            )
            outdim = h

        elif mode == "bertidraaf":
            self.gate = nn.Sequential(
                nn.Linear(4 * h, h),
                nn.LayerNorm(h),
                nn.GELU(),
                nn.Dropout(CONFIG["dropout"]),
                nn.Linear(h, h),
            )
            self.feature_proj = nn.Sequential(
                nn.Linear(8 * h, h),
                nn.LayerNorm(h),
                nn.GELU(),
                nn.Dropout(CONFIG["dropout"]),
                nn.Linear(h, h),
                nn.LayerNorm(h),
                nn.GELU(),
            )
            self.context_head = nn.Linear(2 * h, n_classes)
            self.idf_head = nn.Linear(h, n_classes)
            self.max_head = nn.Linear(h, n_classes)
            self.interaction_head = nn.Linear(2 * h, n_classes)

            initial_mix = inverse_softplus([1.00, 0.15, 0.22, 0.10, 0.08])
            if CONFIG.get("raaf_trainable_head_mixing", True):
                self.head_mix = nn.Parameter(initial_mix)
            else:
                self.register_buffer("head_mix", initial_mix)

            self.msd_dropouts = nn.ModuleList(
                [nn.Dropout(p) for p in CONFIG.get("raaf_msd_dropout_probs", [CONFIG["dropout"]])]
            )
            outdim = h

        else:
            outdim = h

        self.drop = nn.Dropout(CONFIG["dropout"])
        self.fc = nn.Linear(outdim, n_classes)

    def mean(self, x, m):
        m = m.unsqueeze(-1).float()
        return (x * m).sum(1) / m.sum(1).clamp(min=1)

    def mx(self, x, m):
        return x.masked_fill(~m.unsqueeze(-1).bool(), -1e4).max(1).values

    def idfmean(self, x, m, w):
        if w is None:
            return self.mean(x, m)
        ww = (w.to(x.dtype) * m.to(x.dtype)).unsqueeze(-1)
        return (x * ww).sum(1) / ww.sum(1).clamp(min=1e-6)

    def head_fusion_logits(self, feat, cls, mean, idf, mx, drop_layer):
        context_pair = torch.cat([cls, mean], 1)
        interaction_pair = torch.cat([torch.abs(cls - idf), cls * idf], 1)

        logits_list = [
            self.fc(drop_layer(feat)),
            self.context_head(drop_layer(context_pair)),
            self.idf_head(drop_layer(idf)),
            self.max_head(drop_layer(mx)),
            self.interaction_head(drop_layer(interaction_pair)),
        ]

        weights = F.softplus(self.head_mix)
        logits = 0
        for w, head_logits in zip(weights, logits_list):
            logits = logits + w * head_logits
        return logits

    def forward(self, input_ids, attention_mask, labels=None, token_type_ids=None, idf_weights=None):
        kw = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None and self.accepts_token_type_ids:
            kw["token_type_ids"] = token_type_ids

        x = self.encoder(**kw).last_hidden_state

        cls = x[:, 0, :]
        mean = self.mean(x, attention_mask)
        mx = self.mx(x, attention_mask)
        idf = self.idfmean(x, attention_mask, idf_weights)

        if self.mode == "cls":
            feat = cls

        elif self.mode == "mean":
            feat = mean

        elif self.mode == "idf_mean":
            feat = idf

        elif self.mode == "idf_softmax":
            sc = self.attn(x).squeeze(-1)
            if idf_weights is not None:
                sc = sc + torch.log(idf_weights.clamp(min=1e-6)).to(sc.dtype)
            sc = sc.masked_fill(attention_mask == 0, -1e9)
            feat = (x * torch.softmax(sc, 1).unsqueeze(-1)).sum(1)

        elif self.mode == "layer_scale":
            feat = mean * self.scale

        elif self.mode == "cnn":
            xw = x * (idf_weights.unsqueeze(-1).to(x.dtype) if idf_weights is not None else 1)
            z = xw.transpose(1, 2)
            pooled = []
            for conv in self.convs:
                c = F.gelu(conv(z))
                pooled.append(F.max_pool1d(c, kernel_size=c.size(2)).squeeze(2))
            feat = torch.cat(pooled + [cls, mean], 1)

        elif self.mode in {"lstm", "bilstm"}:
            xw = x * (idf_weights.unsqueeze(-1).to(x.dtype) if idf_weights is not None else 1)
            seq_out, _ = self.seq(xw)
            seq_max = seq_out.masked_fill(~attention_mask.unsqueeze(-1).bool(), -1e4).max(1).values
            feat = torch.cat([cls, mean, seq_max], 1)

        elif self.mode == "raif":
            gate = torch.sigmoid(self.gate(torch.cat([cls, mean, mx, idf], 1)))
            fused = gate * idf + (1 - gate) * mean
            feat = self.proj(torch.cat([cls, fused, idf, mx, torch.abs(cls - idf), cls * idf], 1))

        elif self.mode == "bertidraaf":
            gate = torch.sigmoid(self.gate(torch.cat([cls, mean, mx, idf], 1)))
            fused = gate * idf + (1 - gate) * mean

            feat_raw = torch.cat([
                cls,
                mean,
                idf,
                mx,
                fused,
                torch.abs(cls - idf),
                cls * idf,
                fused * mx,
            ], 1)
            feat = self.feature_proj(feat_raw)

            if self.training:
                logits = torch.stack(
                    [self.head_fusion_logits(feat, cls, mean, idf, mx, drop_layer) for drop_layer in self.msd_dropouts],
                    dim=0,
                ).mean(0)
            else:
                logits = self.head_fusion_logits(feat, cls, mean, idf, mx, self.drop)

            loss = F.cross_entropy(logits.float(), labels, label_smoothing=CONFIG["label_smoothing"]) if labels is not None else None
            return {"loss": loss, "logits": logits}

        else:
            raise ValueError(self.mode)

        logits = self.fc(self.drop(feat))
        loss = F.cross_entropy(logits.float(), labels, label_smoothing=CONFIG["label_smoothing"]) if labels is not None else None
        return {"loss": loss, "logits": logits}

def make_loader(df, tok, bundle, batch, max_len, use_idf, shuffle=False):
    return DataLoader(
        TrDataset(df, tok, bundle, max_len, use_idf),
        batch_size=batch,
        shuffle=shuffle,
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
        collate_fn=collate,
    )

def eval_tr(model, loader, bundle):
    model.eval()
    P, y, yp, losses = [], [], [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=torch.cuda.is_available() and getattr(model, "use_autocast", True)):
                out = model(**batch)
            if out["loss"] is not None:
                losses.append(float(out["loss"].detach().cpu()))
            p = torch.softmax(out["logits"].float(), 1).detach().cpu().numpy()
            P.append(p)
            yp += p.argmax(1).tolist()
            y += batch["labels"].detach().cpu().numpy().tolist()

    if not P:
        raise RuntimeError("Empty evaluation loader.")

    P = np.concatenate(P, 0)
    y = np.array(y)
    yp = np.array(yp)
    m = compute_metrics(y, yp, P, bundle["label_order"])
    m.update({"y_true": y, "y_pred": yp, "probas": P, "loss": float(np.mean(losses)) if losses else np.nan})
    return m

def train_transformer(bundle, spec, return_valid=False):
    seed_everything(spec.get("seed", CONFIG["seed"]))
    cleanup()

    tok = load_tokenizer_robust(spec["backbone"])
    max_len = spec.get("max_length", CONFIG["max_length"])
    use_idf = spec.get("use_idf", False)
    batch = spec.get("batch", CONFIG["batch_size_base"])
    grad = spec.get("grad", CONFIG["grad_accum_base"])
    epochs = spec.get("epochs", CONFIG["epochs_transformer"])

    tr = make_loader(bundle["train"], tok, bundle, batch, max_len, use_idf, True)
    va = make_loader(bundle["valid"], tok, bundle, batch * 2, max_len, use_idf)
    te = make_loader(bundle["test"], tok, bundle, batch * 2, max_len, use_idf)

    model = TransformerClassifier(
        spec["backbone"],
        bundle["n_classes"],
        spec.get("mode", "mean"),
        use_idf,
        spec.get("kernels", (2, 3, 4, 5, 6)),
    ).to(device).float()
    model.use_autocast = not bool(spec.get("disable_autocast", False))

    no_decay = ["bias", "LayerNorm.weight", "layer_norm.weight"]
    enc_decay, enc_nodecay, head_decay, head_nodecay = [], [], [], []

    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        is_no = any(nd in name for nd in no_decay)
        if name.startswith("encoder."):
            (enc_nodecay if is_no else enc_decay).append(p)
        else:
            (head_nodecay if is_no else head_decay).append(p)

    groups = []
    if enc_decay:
        groups.append({"params": enc_decay, "lr": spec.get("lr", CONFIG["lr"]), "weight_decay": CONFIG["weight_decay"]})
    if enc_nodecay:
        groups.append({"params": enc_nodecay, "lr": spec.get("lr", CONFIG["lr"]), "weight_decay": 0.0})
    if head_decay:
        groups.append({"params": head_decay, "lr": spec.get("head_lr", CONFIG["head_lr"]), "weight_decay": CONFIG["weight_decay"]})
    if head_nodecay:
        groups.append({"params": head_nodecay, "lr": spec.get("head_lr", CONFIG["head_lr"]), "weight_decay": 0.0})

    opt = torch.optim.AdamW(groups)
    steps = max(int(np.ceil(len(tr) / grad)), 1) * epochs
    sch = get_linear_schedule_with_warmup(opt, max(int(steps * CONFIG["warmup_ratio"]), 1), steps)

    best, best_f1, stale = None, -1, 0
    top_k = int(spec.get("top_k_checkpoints", 1))
    top_states = []
    hist, times = [], []

    for ep in range(1, epochs + 1):
        model.train()
        t0 = time.perf_counter()
        opt.zero_grad(set_to_none=True)
        losses, ty, tp = [], [], []

        for step, batch_data in enumerate(tr, 1):
            batch_data = {k: v.to(device) for k, v in batch_data.items()}
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=torch.cuda.is_available() and getattr(model, "use_autocast", True)):
                out = model(**batch_data)
                loss = out["loss"] / grad

            loss.backward()
            ty += batch_data["labels"].detach().cpu().numpy().tolist()
            tp += out["logits"].detach().argmax(1).cpu().numpy().tolist()

            if step % grad == 0 or step == len(tr):
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
                sch.step()
                opt.zero_grad(set_to_none=True)

            losses.append(float(loss.detach().cpu()) * grad)

        times.append(time.perf_counter() - t0)
        val = eval_tr(model, va, bundle)
        hist.append({
            "epoch": ep,
            "train_loss": float(np.mean(losses)),
            "val_loss": val["loss"],
            "train_acc": accuracy_score(ty, tp),
            "val_acc": val["accuracy"],
            "train_macro_f1": f1_score(ty, tp, average="macro"),
            "val_macro_f1": val["macro_f1"],
        })

        print(f"[{bundle['dataset']}] {spec['name']} epoch {ep}: val_acc={val['accuracy']*100:.2f}, val_f1={val['macro_f1']:.4f}")

        if top_k > 1:
            current_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            top_states.append({
                "macro_f1": float(val["macro_f1"]),
                "accuracy": float(val["accuracy"]),
                "state": current_state,
            })
            top_states = sorted(top_states, key=lambda x: (x["macro_f1"], x["accuracy"]), reverse=True)[:top_k]

        if val["macro_f1"] > best_f1:
            best_f1 = val["macro_f1"]
            best = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stale = 0
        else:
            stale += 1

        if stale >= spec.get("patience", CONFIG["early_stopping_patience"]):
            break

    if top_k > 1 and top_states:
        valid_probas_list, test_probas_list = [], []
        valid_y_true, test_y_true = None, None
        t = time.perf_counter()

        for item in top_states:
            model.load_state_dict(item["state"])
            valid_i = eval_tr(model, va, bundle)
            test_i = eval_tr(model, te, bundle)
            valid_probas_list.append(valid_i["probas"])
            test_probas_list.append(test_i["probas"])
            valid_y_true = valid_i["y_true"]
            test_y_true = test_i["y_true"]

        valid_probas = np.mean(valid_probas_list, axis=0)
        test_probas = np.mean(test_probas_list, axis=0)
        valid_pred = valid_probas.argmax(axis=1)
        test_pred = test_probas.argmax(axis=1)

        valid = compute_metrics(valid_y_true, valid_pred, valid_probas, bundle["label_order"])
        valid.update({"y_true": valid_y_true, "y_pred": valid_pred, "probas": valid_probas, "loss": np.nan})

        test = compute_metrics(test_y_true, test_pred, test_probas, bundle["label_order"])
        test.update({"y_true": test_y_true, "y_pred": test_pred, "probas": test_probas, "loss": np.nan})
        infer = time.perf_counter() - t
        checkpoint_ensemble = True

    else:
        model.load_state_dict(best)
        valid = eval_tr(model, va, bundle)

        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

        t = time.perf_counter()
        test = eval_tr(model, te, bundle)
        infer = time.perf_counter() - t
        checkpoint_ensemble = False

    res = {
        "dataset": bundle["dataset"],
        "model": spec["name"],
        "family": spec["family"],
        "history": hist,
        "train_time_epoch_s": float(np.mean(times)),
        "inference_total_s": infer,
        "inference_ms_per_doc": infer / max(len(bundle["test"]), 1) * 1000,
        "params": count_parameters(model),
        "spec": spec,
        "checkpoint_ensemble": checkpoint_ensemble,
        "top_k_checkpoints": top_k,
        **test,
    }

    if return_valid:
        res["valid_probas"] = valid["probas"]
        res["valid_y_true"] = valid["y_true"]

    return res

def temp_probs(p, t):
    return safe_proba(np.log(np.asarray(p).clip(min=1e-12)) / t)

def lexical_branch(bundle):
    Xtr, Xva, Xte, ytr, yva, yte = tfidf_matrices(bundle)

    lr = LogisticRegression(C=4.0, max_iter=3000, n_jobs=-1)
    lr.fit(Xtr, ytr)
    vlr = lr.predict_proba(Xva)
    tlr = lr.predict_proba(Xte)

    svm = CalibratedClassifierCV(LinearSVC(), cv=3)
    svm.fit(Xtr, ytr)
    vsvm = svm.predict_proba(Xva)
    tsvm = svm.predict_proba(Xte)

    nb = MultinomialNB(alpha=0.05)
    nb.fit(Xtr, ytr)
    vnb = nb.predict_proba(Xva)
    tnb = nb.predict_proba(Xte)

    Xn = normalize(Xtr)
    Xv = normalize(Xva)
    Xt = normalize(Xte)

    cent = []
    for c in range(bundle["n_classes"]):
        rows = Xn[ytr == c]
        cent.append(np.asarray(rows.mean(axis=0)).ravel() if rows.shape[0] else np.zeros(Xn.shape[1]))
    cent = normalize(np.vstack(cent))

    vc = safe_proba(Xv @ cent.T)
    tc = safe_proba(Xt @ cent.T)

    return {
        "valid_lr": vlr,
        "test_lr": tlr,
        "valid_svm": vsvm,
        "test_svm": tsvm,
        "valid_centroid": vc,
        "test_centroid": tc,
        "valid_nb": vnb,
        "test_nb": tnb,
        "y_valid": yva,
    }

def temp_probs(p, t):
    return safe_proba(np.log(np.asarray(p).clip(min=1e-12)) / t)

def blend_probabilities(weight_info, bert_p, logreg_p, svm_p, centroid_p, nb_p):
    sources = [
        (bert_p, weight_info["w_bert"], weight_info["temp_bert"]),
        (logreg_p, weight_info["w_logreg"], weight_info["temp_logreg"]),
        (svm_p, weight_info["w_svm"], weight_info["temp_svm"]),
        (centroid_p, weight_info["w_centroid"], weight_info["temp_centroid"]),
        (nb_p, weight_info["w_nb"], weight_info["temp_nb"]),
    ]

    calibrated = [(temp_probs(p, t), w) for p, w, t in sources]

    if weight_info.get("fusion_type", "linear") == "geometric":
        log_mix = None
        for probs, w in calibrated:
            if w <= 0:
                continue
            component = w * np.log(np.asarray(probs).clip(min=1e-12))
            log_mix = component if log_mix is None else log_mix + component
        return safe_proba(log_mix)

    return sum(w * probs for probs, w in calibrated)

def lexical_average_probs(lr_p, svm_p, centroid_p, nb_p):
    return 0.35 * lr_p + 0.35 * svm_p + 0.15 * centroid_p + 0.15 * nb_p

def probability_features(*prob_list):
    parts = []
    for p in prob_list:
        p = np.asarray(p, dtype=np.float64).clip(min=1e-12, max=1.0)
        parts.append(p)
        parts.append(np.log(p))
    return np.hstack(parts)

def evaluate_candidate_on_validation(y, probs=None, pred=None):
    if pred is None:
        pred = np.asarray(probs).argmax(axis=1)
    return {
        "accuracy": float(accuracy_score(y, pred)),
        "macro_f1": float(f1_score(y, pred, average="macro")),
    }

def override_with_idf_expert(bert_valid, expert_valid, bert_test, expert_test, tau, margin):
    bert_valid = np.asarray(bert_valid)
    expert_valid = np.asarray(expert_valid)
    bert_test = np.asarray(bert_test)
    expert_test = np.asarray(expert_test)

    valid_out = bert_valid.copy()
    test_out = bert_test.copy()

    valid_expert_conf = expert_valid.max(axis=1)
    valid_bert_conf = bert_valid.max(axis=1)
    valid_mask = (valid_expert_conf >= tau) & ((valid_expert_conf - valid_bert_conf) >= margin)
    valid_out[valid_mask] = expert_valid[valid_mask]

    test_expert_conf = expert_test.max(axis=1)
    test_bert_conf = bert_test.max(axis=1)
    test_mask = (test_expert_conf >= tau) & ((test_expert_conf - test_bert_conf) >= margin)
    test_out[test_mask] = expert_test[test_mask]

    return valid_out, test_out

def majority_vote_probs(prob_sources):
    preds = [np.asarray(p).argmax(axis=1) for p in prob_sources]
    n = prob_sources[0].shape[0]
    c = prob_sources[0].shape[1]
    votes = np.zeros((n, c), dtype=np.float64)
    for pred in preds:
        votes[np.arange(n), pred] += 1.0
    votes = votes / len(preds)
    avg_probs = sum(np.asarray(p) for p in prob_sources) / len(prob_sources)
    return 0.50 * votes + 0.50 * avg_probs

def make_binary_threshold_candidate(valid_probs, test_probs, y_valid, thresholds):
    valid_probs = np.asarray(valid_probs)
    test_probs = np.asarray(test_probs)

    if valid_probs.shape[1] != 2:
        return None

    best = None
    for target_class in [0, 1]:
        other_class = 1 - target_class
        valid_scores = valid_probs[:, target_class]

        for threshold in thresholds:
            valid_pred = np.where(valid_scores >= threshold, target_class, other_class)
            metrics = evaluate_candidate_on_validation(y_valid, pred=valid_pred)

            if best is None or (metrics["macro_f1"], metrics["accuracy"]) > (best["macro_f1"], best["accuracy"]):
                test_pred = np.where(test_probs[:, target_class] >= threshold, target_class, other_class)
                best = {
                    "valid_probs": valid_probs,
                    "test_probs": test_probs,
                    "valid_pred": valid_pred,
                    "test_pred": test_pred,
                    "threshold": float(threshold),
                    "target_class": int(target_class),
                    **metrics,
                }

    return best

def tune_fusion(y, bv, lv, sv, cv, nv, dataset_name=None):
    cfg = CONFIG["dataset_model_config"].get(dataset_name or "", {})
    min_w_bert = float(cfg.get("min_w_bert", 0.10))
    min_w_logreg = float(cfg.get("min_w_logreg", 0.00))
    min_w_svm = float(cfg.get("min_w_svm", 0.00))
    min_w_centroid = float(cfg.get("min_w_centroid", 0.00))
    min_w_nb = float(cfg.get("min_w_nb", 0.00))

    grid = np.arange(0, 1 + 1e-9, CONFIG["fusion_weight_grid_step"])
    temps = CONFIG["fusion_temperature_grid"]
    fusion_types = ["linear", "geometric"] if CONFIG.get("allow_geometric_fusion", True) else ["linear"]
    best = {"macro_f1": -1}

    for tb in temps:
        for tl in temps:
            for wb in grid:
                if wb < min_w_bert:
                    continue
                for wl in grid:
                    if wl < min_w_logreg:
                        continue
                    for ws in grid:
                        if ws < min_w_svm:
                            continue
                        for wc in grid:
                            if wc < min_w_centroid:
                                continue
                            wn = 1.0 - wb - wl - ws - wc
                            if wn < -1e-9:
                                continue
                            wn = max(0.0, wn)
                            if wn < min_w_nb:
                                continue

                            base = {
                                "w_bert": float(wb),
                                "w_logreg": float(wl),
                                "w_svm": float(ws),
                                "w_centroid": float(wc),
                                "w_nb": float(wn),
                                "temp_bert": float(tb),
                                "temp_logreg": float(tl),
                                "temp_svm": 1.0,
                                "temp_centroid": 1.0,
                                "temp_nb": 1.0,
                                "min_w_bert_constraint": min_w_bert,
                            }

                            for fusion_type in fusion_types:
                                info = dict(base)
                                info["fusion_type"] = fusion_type
                                P = blend_probabilities(info, bv, lv, sv, cv, nv)
                                metrics = evaluate_candidate_on_validation(y, P)
                                if (metrics["macro_f1"], metrics["accuracy"]) > (best["macro_f1"], best.get("accuracy", -1)):
                                    best = dict(info)
                                    best.update(metrics)

    if best["macro_f1"] < 0:
        raise RuntimeError("No valid fusion weights found.")
    return best

def confidence_router_probs(bert_p, lexical_p, tau=0.75, alpha=0.50):
    bert_p = np.asarray(bert_p)
    lexical_p = np.asarray(lexical_p)
    lexical_conf = lexical_p.max(axis=1)

    routed = alpha * bert_p + (1.0 - alpha) * lexical_p
    high = lexical_conf >= tau
    routed[high] = 0.10 * bert_p[high] + 0.90 * lexical_p[high]
    return routed

def fit_meta_calibrator(y_valid, valid_sources, test_sources):
    X_valid = probability_features(*valid_sources)
    X_test = probability_features(*test_sources)
    n_total = valid_sources[0].shape[1]

    clf = LogisticRegression(C=0.50, max_iter=2000, class_weight="balanced" if n_total == 2 else None, n_jobs=-1)
    clf.fit(X_valid, y_valid)

    valid_probs = clf.predict_proba(X_valid)
    test_probs = clf.predict_proba(X_test)

    if valid_probs.shape[1] != n_total or not np.array_equal(clf.classes_.astype(int), np.arange(n_total)):
        aligned_valid = np.zeros((valid_probs.shape[0], n_total), dtype=np.float64)
        aligned_test = np.zeros((test_probs.shape[0], n_total), dtype=np.float64)
        for col, cls in enumerate(clf.classes_.astype(int)):
            aligned_valid[:, cls] = valid_probs[:, col]
            aligned_test[:, cls] = test_probs[:, col]
        valid_probs, test_probs = aligned_valid, aligned_test

    return valid_probs, test_probs

def average_seed_outputs(seed_outputs):
    base = seed_outputs[0]
    valid_stack = np.stack([r["valid_probas"] for r in seed_outputs], axis=0)
    test_stack = np.stack([r["probas"] for r in seed_outputs], axis=0)

    merged = dict(base)
    merged["valid_probas"] = valid_stack.mean(axis=0)
    merged["probas"] = test_stack.mean(axis=0)
    merged["history"] = base.get("history", [])
    merged["train_time_epoch_s"] = float(np.mean([r.get("train_time_epoch_s", np.nan) for r in seed_outputs]))
    merged["inference_total_s"] = float(np.mean([r.get("inference_total_s", np.nan) for r in seed_outputs]))
    merged["inference_ms_per_doc"] = float(np.mean([r.get("inference_ms_per_doc", np.nan) for r in seed_outputs]))
    merged["params"] = base.get("params", np.nan)
    return merged

def select_best_candidate(y_valid, candidates):
    best_name, best_info = None, None
    for name, info in candidates.items():
        metrics = evaluate_candidate_on_validation(
            y_valid,
            info.get("valid_probs"),
            info.get("valid_pred"),
        )
        info.update(metrics)
        if best_info is None or (metrics["macro_f1"], metrics["accuracy"]) > (best_info["macro_f1"], best_info["accuracy"]):
            best_name, best_info = name, info
    return best_name, best_info, candidates

def run_raaf(bundle):
    dcfg = CONFIG["dataset_model_config"].get(bundle["dataset"], {})
    backbone = dcfg.get("backbone", "bert-large-uncased" if is_h100 else "bert-base-uncased")
    if (not is_h100) and "large" in backbone:
        backbone = "bert-base-uncased"

    seeds = dcfg.get("proposed_seeds", [CONFIG["seed"]])
    seed_outputs = []

    for seed in seeds:
        spec = {
            "name": CONFIG["proposed_model_name"],
            "family": "Proposed BERT+IDF BERTidRAAF",
            "backbone": backbone,
            "mode": "bertidraaf",
            "use_idf": True,
            "max_length": dcfg.get("max_length", CONFIG["proposed_max_length"]),
            "batch": CONFIG["batch_size_large"] if "large" in backbone else CONFIG["batch_size_base"],
            "grad": CONFIG["grad_accum_large"] if "large" in backbone else CONFIG["grad_accum_base"],
            "epochs": dcfg.get("epochs", CONFIG["epochs_proposed"]),
            "lr": dcfg.get("lr", CONFIG["lr"]),
            "head_lr": dcfg.get("head_lr", CONFIG["head_lr"]),
            "seed": seed,
            "top_k_checkpoints": dcfg.get("top_k_checkpoints", 1),
            "patience": dcfg.get("patience", CONFIG["early_stopping_patience"]),
        }
        seed_outputs.append(train_transformer(bundle, spec, return_valid=True))

    core = average_seed_outputs(seed_outputs)
    lex = lexical_branch(bundle)

    w = tune_fusion(
        core["valid_y_true"],
        core["valid_probas"],
        lex["valid_lr"],
        lex["valid_svm"],
        lex["valid_centroid"],
        lex["valid_nb"],
        dataset_name=bundle["dataset"],
    )

    valid_fused = blend_probabilities(w, core["valid_probas"], lex["valid_lr"], lex["valid_svm"], lex["valid_centroid"], lex["valid_nb"])
    test_fused = blend_probabilities(w, core["probas"], lex["test_lr"], lex["test_svm"], lex["test_centroid"], lex["test_nb"])

    valid_lexavg = lexical_average_probs(lex["valid_lr"], lex["valid_svm"], lex["valid_centroid"], lex["valid_nb"])
    test_lexavg = lexical_average_probs(lex["test_lr"], lex["test_svm"], lex["test_centroid"], lex["test_nb"])

    policy = dcfg.get("candidate_policy", "bertidf_validation_selection")

    candidates = {
        "BERT-IDF rare-aware core": {
            "valid_probs": core["valid_probas"],
            "test_probs": core["probas"],
        },
        "BERT-IDF validation fusion": {
            "valid_probs": valid_fused,
            "test_probs": test_fused,
        },
    }

    if policy == "core_preservation":
        selected_name = "BERT-IDF rare-aware core"
        selected_info = candidates[selected_name]
        candidates = {selected_name: selected_info}
    else:
        lexical_sources = {
            "LogReg": (lex["valid_lr"], lex["test_lr"]),
            "SVM": (lex["valid_svm"], lex["test_svm"]),
            "Centroid": (lex["valid_centroid"], lex["test_centroid"]),
            "NB": (lex["valid_nb"], lex["test_nb"]),
            "LexicalAvg": (valid_lexavg, test_lexavg),
        }

        for beta in dcfg.get("bert_anchor_grid", [0.10, 0.20, 0.30]):
            for source_name, (valid_source, test_source) in lexical_sources.items():
                candidates[f"BERT-anchored IDF {source_name} beta={beta:.2f}"] = {
                    "valid_probs": beta * core["valid_probas"] + (1.0 - beta) * valid_source,
                    "test_probs": beta * core["probas"] + (1.0 - beta) * test_source,
                    "beta": float(beta),
                    "source": source_name,
                }

        for tau in dcfg.get("confidence_tau_grid", [0.70, 0.80]):
            for alpha in dcfg.get("confidence_alpha_grid", [0.35, 0.50]):
                for source_name, (valid_source, test_source) in lexical_sources.items():
                    candidates[f"Confidence-routed BERT-IDF {source_name} tau={tau:.2f} alpha={alpha:.2f}"] = {
                        "valid_probs": confidence_router_probs(core["valid_probas"], valid_source, tau=float(tau), alpha=float(alpha)),
                        "test_probs": confidence_router_probs(core["probas"], test_source, tau=float(tau), alpha=float(alpha)),
                        "tau": float(tau),
                        "alpha": float(alpha),
                        "source": source_name,
                    }

        try:
            valid_meta, test_meta = fit_meta_calibrator(
                core["valid_y_true"],
                [core["valid_probas"], lex["valid_lr"], lex["valid_svm"], lex["valid_centroid"], lex["valid_nb"]],
                [core["probas"], lex["test_lr"], lex["test_svm"], lex["test_centroid"], lex["test_nb"]],
            )
            candidates["Validation-trained BERT-IDF meta-calibrator"] = {
                "valid_probs": valid_meta,
                "test_probs": test_meta,
            }
        except Exception as exc:
            print(f"[{bundle['dataset']}] Meta-calibrator skipped:", repr(exc))

        selected_name, selected_info, candidates = select_best_candidate(core["valid_y_true"], candidates)

    for name, info in candidates.items():
        if "macro_f1" not in info:
            info.update(
                evaluate_candidate_on_validation(
                    core["valid_y_true"],
                    info.get("valid_probs"),
                    info.get("valid_pred"),
                )
            )

    P = selected_info["test_probs"]
    yp = selected_info.get("test_pred", P.argmax(1))
    m = compute_metrics(core["y_true"], yp, P, bundle["label_order"])

    candidate_rows = []
    for name, info in candidates.items():
        candidate_rows.append({
            "Candidate": name,
            "Validation Accuracy": info.get("accuracy", np.nan),
            "Validation Macro-F1": info.get("macro_f1", np.nan),
            "tau": info.get("tau", np.nan),
            "alpha": info.get("alpha", np.nan),
            "beta": info.get("beta", np.nan),
            "source": info.get("source", np.nan),
            "threshold": info.get("threshold", np.nan),
            "target_class": info.get("target_class", np.nan),
            "margin": info.get("margin", np.nan),
            "margin": info.get("margin", np.nan),
        })

    core.update(m)
    core.update({
        "y_pred": yp,
        "probas": P,
        "fusion_weights": w,
        "selected_prediction_strategy": selected_name,
        "candidate_policy": policy,
        "candidate_validation_table": pd.DataFrame(candidate_rows).sort_values(
            ["Validation Macro-F1", "Validation Accuracy"], ascending=False
        ),
        "validation_core_macro_f1": float(evaluate_candidate_on_validation(core["valid_y_true"], core["valid_probas"])["macro_f1"]),
        "validation_selected_macro_f1": float(selected_info.get("macro_f1", np.nan)),
        "validation_selected_accuracy": float(selected_info.get("accuracy", np.nan)),
        "model": CONFIG["proposed_model_name"],
        "family": "Proposed BERT+IDF BERTidRAAF",
        "spec": {
            "name": CONFIG["proposed_model_name"],
            "family": "Proposed BERT+IDF BERTidRAAF",
            "backbone": backbone,
            "mode": "bertidraaf",
            "use_idf": True,
            "seeds": seeds,
        },
    })

    print(f"[{bundle['dataset']}] Selected BERT-IDF strategy:", selected_name)
    display(core["candidate_validation_table"])
    return core


## 7. Model registry


In [ ]:
TRANSFORMER_SPECS = [
    # 3. Transformer contextual baselines.
    {"name": "BERTBASE", "family": "Transformer Context Baseline", "backbone": "bert-base-uncased", "mode": "mean", "use_idf": False},
    {"name": "BERTLARGE", "family": "Transformer Context Baseline", "backbone": "bert-large-uncased", "mode": "mean", "use_idf": False, "batch": CONFIG["batch_size_large"], "grad": CONFIG["grad_accum_large"]},
    {"name": "RoBERTaBASE", "family": "Transformer Context Baseline", "backbone": "roberta-base", "mode": "mean", "use_idf": False},
    {"name": "DeBERTaBASE", "family": "Transformer Context Baseline", "backbone": "microsoft/deberta-base", "mode": "mean", "use_idf": False, "max_length": 256, "batch": 4 if is_h100 else 1, "grad": 4 if is_h100 else 16, "disable_autocast": True},

    # 4. Closest hybrids to the proposed contribution.
    {"name": "BERT + IDF", "family": "BERT–IDF Hybrid/Ablation", "backbone": "bert-base-uncased", "mode": "idf_mean", "use_idf": True},
    {"name": "BERT + CNN", "family": "BERT–IDF Hybrid/Ablation", "backbone": "bert-base-uncased", "mode": "cnn", "use_idf": False},
    {"name": "BERT–IDF + CNN", "family": "BERT–IDF Hybrid/Ablation", "backbone": "bert-base-uncased", "mode": "cnn", "use_idf": True},
    {"name": "BERT–IDF + LSTM", "family": "BERT–IDF Hybrid/Ablation", "backbone": "bert-base-uncased", "mode": "lstm", "use_idf": True},
    {"name": "BERT–IDF + BiLSTM", "family": "BERT–IDF Hybrid/Ablation", "backbone": "bert-base-uncased", "mode": "bilstm", "use_idf": True},
]

MODEL_REGISTRY_TABLE = pd.DataFrame(
    [{"Stage": "Proposed", "Model": CONFIG["proposed_model_name"], "Family": "Proposed BERT+IDF BERTidRAAF", "Backbone": "bert-large-uncased", "Purpose": "Final rare-aware adaptive fusion model"}]
    + [{"Stage": "1. Lexical", "Model": m, "Family": "Lexical Baseline", "Backbone": "TF-IDF", "Purpose": "Keyword-only baseline"} for m in CLASSICAL_MODELS]
    + [{"Stage": "2. Neural", "Model": "CNN", "Family": "Neural Baseline", "Backbone": "Embedding + CNN", "Purpose": "Convolutional non-transformer baseline"}]
    + [{"Stage": "2. Neural", "Model": m, "Family": "Neural Baseline", "Backbone": "Embedding + LSTM", "Purpose": "Sequential non-transformer baseline"} for m in RECURRENT_MODELS]
    + [{"Stage": "3. Transformer", "Model": s["name"], "Family": s["family"], "Backbone": s["backbone"], "Purpose": "Contextual semantics baseline"} for s in TRANSFORMER_SPECS[:4]]
    + [{"Stage": "4. BERT–IDF hybrids", "Model": s["name"], "Family": s["family"], "Backbone": s["backbone"], "Purpose": "Closest ablation/comparator to BERTidRAAF"} for s in TRANSFORMER_SPECS[4:]]
)

display(MODEL_REGISTRY_TABLE)


## 8. Run benchmark on FairCVdb and Resume


In [ ]:
ALL_RESULTS = []
FAILED_RUNS = []

def record_failure(dataset, model, exc):
    FAILED_RUNS.append({"Dataset": dataset, "Model": model, "Error": repr(exc)})
    print(f"[{dataset}] {model} failed but the benchmark continues:", repr(exc))
    cleanup()

for dname, bundle in DATASETS.items():
    print("\n" + "=" * 90 + f"\nDATASET: {dname}\n" + "=" * 90)

    try:
        r = run_raaf(bundle)
        ALL_RESULTS.append(r)
        print(f"[{dname}] {CONFIG['proposed_model_name']}: acc={r['accuracy']*100:.2f}, f1={r['macro_f1']:.4f}")

    except Exception as e:
        record_failure(dname, CONFIG["proposed_model_name"], e)

    if CONFIG["run_classical_models"]:
        for m in CLASSICAL_MODELS:
            try:
                r = run_classical(bundle, m)
                ALL_RESULTS.append(r)
                print(f"[{dname}] {m}: acc={r['accuracy']*100:.2f}, f1={r['macro_f1']:.4f}")
            except Exception as e:
                record_failure(dname, m, e)

    if CONFIG["run_convolutional_models"]:
        for m in CONVOLUTIONAL_MODELS:
            try:
                r = run_convolutional(bundle, m)
                ALL_RESULTS.append(r)
                print(f"[{dname}] {m}: acc={r['accuracy']*100:.2f}, f1={r['macro_f1']:.4f}")
            except Exception as e:
                record_failure(dname, m, e)

    if CONFIG["run_recurrent_models"]:
        for m in RECURRENT_MODELS:
            try:
                r = run_recurrent(bundle, m)
                ALL_RESULTS.append(r)
                print(f"[{dname}] {m}: acc={r['accuracy']*100:.2f}, f1={r['macro_f1']:.4f}")
            except Exception as e:
                record_failure(dname, m, e)

    if CONFIG["run_transformer_models"]:
        for s in TRANSFORMER_SPECS:
            try:
                r = train_transformer(bundle, s)
                ALL_RESULTS.append(r)
                print(f"[{dname}] {s['name']}: acc={r['accuracy']*100:.2f}, f1={r['macro_f1']:.4f}")
            except Exception as e:
                record_failure(dname, s["name"], e)

print("Total successful results:", len(ALL_RESULTS))
if FAILED_RUNS:
    print("Failed runs were recorded without stopping the notebook:")
    display(pd.DataFrame(FAILED_RUNS))


## 9. Benchmark and article-ready tables


In [ ]:
def row(r):
    _, acc_lo, acc_hi = bootstrap_accuracy_ci(r["y_true"], r["y_pred"])
    _, f1_lo, f1_hi = bootstrap_macro_f1_ci(r["y_true"], r["y_pred"])
    return {
        "Dataset": r["dataset"],
        "Model": r["model"],
        "Family": r["family"],
        "Accuracy (%)": round(r["accuracy"] * 100, 2),
        "Macro-F1": round(r["macro_f1"], 4),
        "Weighted-F1": round(r["weighted_f1"], 4),
        "Macro Precision": round(r["macro_precision"], 4),
        "Macro Recall": round(r["macro_recall"], 4),
        "Micro-F1": round(r["micro_f1"], 4),
        "AUC-macro": round(r["auc_macro"], 4) if not pd.isna(r["auc_macro"]) else np.nan,
        "Accuracy 95% CI": f"{acc_lo*100:.2f}–{acc_hi*100:.2f}",
        "Macro-F1 95% CI": f"{f1_lo:.4f}–{f1_hi:.4f}",
        "Train Time / Epoch (s)": round(r.get("train_time_epoch_s", np.nan), 2) if not pd.isna(r.get("train_time_epoch_s", np.nan)) else np.nan,
        "Inference ms/document": round(r.get("inference_ms_per_doc", np.nan), 3) if not pd.isna(r.get("inference_ms_per_doc", np.nan)) else np.nan,
        "Trainable Params": r.get("params", np.nan),
    }

T03_compact_cross_dataset_benchmark = pd.DataFrame([row(r) for r in ALL_RESULTS])
if not T03_compact_cross_dataset_benchmark.empty:
    T03_compact_cross_dataset_benchmark = T03_compact_cross_dataset_benchmark.sort_values(
        ["Dataset", "Accuracy (%)", "Macro-F1"], ascending=[True, False, False]
    )

def family_summary():
    rows = []
    for (d, f), g in T03_compact_cross_dataset_benchmark.groupby(["Dataset", "Family"]):
        best = g.sort_values(["Accuracy (%)", "Macro-F1"], ascending=False).iloc[0]
        rows.append({
            "Dataset": d,
            "Family": f,
            "Models": ", ".join(g.Model.tolist()),
            "Accuracy Range (%)": f"{g['Accuracy (%)'].min():.2f}–{g['Accuracy (%)'].max():.2f}",
            "Macro-F1 Range": f"{g['Macro-F1'].min():.4f}–{g['Macro-F1'].max():.4f}",
            "Best Model in Family": best.Model,
        })
    return pd.DataFrame(rows)

T04_model_family_summary = family_summary() if not T03_compact_cross_dataset_benchmark.empty else pd.DataFrame()

def gains():
    rows = []
    for d, g in T03_compact_cross_dataset_benchmark.groupby("Dataset"):
        p = g[g.Model == CONFIG["proposed_model_name"]]
        if p.empty:
            continue
        p = p.iloc[0]
        b = g[g.Model != CONFIG["proposed_model_name"]].sort_values(["Accuracy (%)", "Macro-F1"], ascending=False)
        if b.empty:
            continue
        b = b.iloc[0]
        rows.append({
            "Dataset": d,
            "Proposed Model": p.Model,
            "Proposed Accuracy (%)": p["Accuracy (%)"],
            "Proposed Macro-F1": p["Macro-F1"],
            "Best Baseline": b.Model,
            "Best Baseline Accuracy (%)": b["Accuracy (%)"],
            "Best Baseline Macro-F1": b["Macro-F1"],
            "Accuracy Gain (pp)": round(p["Accuracy (%)"] - b["Accuracy (%)"], 2),
            "Macro-F1 Gain": round(p["Macro-F1"] - b["Macro-F1"], 4),
            "Is Proposed Top?": bool(p["Accuracy (%)"] >= b["Accuracy (%)"]),
        })
    return pd.DataFrame(rows)

T05_proposed_vs_best_baseline = gains()

T06_macro_micro_metrics = T03_compact_cross_dataset_benchmark[
    ["Dataset", "Model", "Family", "Macro-F1", "Macro-F1 95% CI", "Micro-F1", "Macro Precision", "Macro Recall", "Weighted-F1", "AUC-macro"]
].copy() if not T03_compact_cross_dataset_benchmark.empty else pd.DataFrame()

def perclass():
    rows = []
    for r in ALL_RESULTS:
        if r["model"] != CONFIG["proposed_model_name"]:
            continue
        for cls in DATASETS[r["dataset"]]["label_order"]:
            rep = r["report"][cls]
            rows.append({
                "Dataset": r["dataset"],
                "Class": cls,
                "Precision": round(rep["precision"], 4),
                "Recall": round(rep["recall"], 4),
                "F1-score": round(rep["f1-score"], 4),
                "Support": int(rep["support"]),
            })
    return pd.DataFrame(rows)

T07_per_class_proposed = perclass()

def confpairs(k=15):
    rows = []
    for r in ALL_RESULTS:
        if r["model"] != CONFIG["proposed_model_name"]:
            continue
        labs = DATASETS[r["dataset"]]["label_order"]
        cm = confusion_matrix(r["y_true"], r["y_pred"], labels=np.arange(len(labs)))
        for i in range(len(labs)):
            for j in range(len(labs)):
                if i != j and cm[i, j] > 0:
                    rows.append({"Dataset": r["dataset"], "True Class": labs[i], "Predicted Class": labs[j], "Count": int(cm[i, j])})
    f = pd.DataFrame(rows)
    return f.sort_values(["Dataset", "Count"], ascending=[True, False]).groupby("Dataset").head(k) if not f.empty else f

T08_error_analysis_confusion_pairs = confpairs()

T09_efficiency_profile = T03_compact_cross_dataset_benchmark[
    ["Dataset", "Model", "Family", "Accuracy (%)", "Macro-F1", "Train Time / Epoch (s)", "Inference ms/document", "Trainable Params"]
].copy() if not T03_compact_cross_dataset_benchmark.empty else pd.DataFrame()

hybrid_keep = [
    CONFIG["proposed_model_name"],
    "BERT + IDF",
    "BERT + CNN",
    "BERT–IDF + CNN",
    "BERT–IDF + LSTM",
    "BERT–IDF + BiLSTM",
]
T10_bertidf_hybrid_ablation = T03_compact_cross_dataset_benchmark[
    T03_compact_cross_dataset_benchmark.Model.isin(hybrid_keep)
].copy() if not T03_compact_cross_dataset_benchmark.empty else pd.DataFrame()

def fusion_table():
    rows = []
    for r in ALL_RESULTS:
        if r["model"] != CONFIG["proposed_model_name"]:
            continue
        w = r.get("fusion_weights", {})
        rows.append({
            "Dataset": r["dataset"],
            "Model": r["model"],
            "Selected Strategy": r.get("selected_prediction_strategy", np.nan),
            "Candidate Policy": r.get("candidate_policy", np.nan),
            "Proposed Backbone": r.get("spec", {}).get("backbone", np.nan),
            "Proposed Seeds": r.get("spec", {}).get("seeds", np.nan),
            "Validation Selected Accuracy": r.get("validation_selected_accuracy", np.nan),
            "Validation Selected Macro-F1": r.get("validation_selected_macro_f1", np.nan),
            "Validation Core Macro-F1": r.get("validation_core_macro_f1", np.nan),
            "Fusion Type": w.get("fusion_type", np.nan),
            "w_BERT": w.get("w_bert", np.nan),
            "w_LogReg": w.get("w_logreg", np.nan),
            "w_SVM": w.get("w_svm", np.nan),
            "w_Centroid": w.get("w_centroid", np.nan),
            "w_NB": w.get("w_nb", np.nan),
        })
    return pd.DataFrame(rows)

T11_fusion_weights = fusion_table()

T12_failed_runs = pd.DataFrame(FAILED_RUNS) if "FAILED_RUNS" in globals() and FAILED_RUNS else pd.DataFrame(columns=["Dataset", "Model", "Error"])

def proposed_candidate_strategy_table():
    rows = []
    for r in ALL_RESULTS:
        if r["model"] != CONFIG["proposed_model_name"]:
            continue
        frame = r.get("candidate_validation_table")
        if isinstance(frame, pd.DataFrame):
            tmp = frame.copy()
            tmp.insert(0, "Dataset", r["dataset"])
            rows.append(tmp)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

T13_candidate_strategy_selection = proposed_candidate_strategy_table()

T14_model_selection_rationale = pd.DataFrame([
    {"Stage": "Lexical baselines", "Models": "TF-IDF + SVM, LogReg, Naive Bayes", "Rationale": "Measures keyword-only evidence."},
    {"Stage": "Neural baselines", "Models": "CNN, LSTM, BiLSTM", "Rationale": "Measures non-transformer sequence modeling."},
    {"Stage": "Transformer baselines", "Models": "BERTBASE, BERTLARGE, RoBERTaBASE, DeBERTaBASE", "Rationale": "Measures contextual semantics without explicit IDF fusion."},
    {"Stage": "BERT–IDF hybrids", "Models": "BERT+IDF, BERT+CNN, BERT–IDF+CNN, BERT–IDF+LSTM, BERT–IDF+BiLSTM", "Rationale": "Tests whether the proposed fusion is stronger than simpler BERT–IDF designs."},
    {"Stage": "Proposed", "Models": "BERTidRAAF (Ours)", "Rationale": "Combines BERT context, IDF token weighting, rare-aware gated pooling, multi-head logits, and validation-selected BERT–IDF expert fusion."},
])

ARTICLE_TABLES = {
    "T00_model_registry": MODEL_REGISTRY_TABLE,
    "T01_dataset_accounting": T01_dataset_accounting,
    "T02_label_distribution": T02_label_distribution,
    "T03_compact_cross_dataset_benchmark": T03_compact_cross_dataset_benchmark,
    "T04_model_family_summary": T04_model_family_summary,
    "T05_proposed_vs_best_baseline": T05_proposed_vs_best_baseline,
    "T06_macro_micro_metrics": T06_macro_micro_metrics,
    "T07_per_class_proposed": T07_per_class_proposed,
    "T08_error_analysis_confusion_pairs": T08_error_analysis_confusion_pairs,
    "T09_efficiency_profile": T09_efficiency_profile,
    "T10_bertidf_hybrid_ablation": T10_bertidf_hybrid_ablation,
    "T11_fusion_weights": T11_fusion_weights,
    "T12_failed_runs": T12_failed_runs,
    "T13_candidate_strategy_selection": T13_candidate_strategy_selection,
    "T14_model_selection_rationale": T14_model_selection_rationale,
}

for n, t in ARTICLE_TABLES.items():
    print("\n", n)
    display(t.head(30))


## 10. Figures

This section generates publication-ready figures for:

1. cross-model performance comparisons;
2. training curves for neural, transformer, hybrid, and proposed-model groups;
3. grouped confusion matrices by model family;
4. standalone confusion matrices for the proposed model;
5. category-wise ROC visualizations and standalone ROC plots for the proposed model;
6. bootstrap confidence-interval plots for test accuracy and macro-F1;
7. efficiency plots.

In [ ]:
from pathlib import Path
import string
import textwrap
import itertools
from mpl_toolkits.axes_grid1 import make_axes_locatable
from sklearn.metrics import confusion_matrix, roc_curve, auc

FIG_DIR = Path(CONFIG["output_dir"]) / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

def sname(x):
    return re.sub(r"[^A-Za-z0-9_\-]+", "_", str(x)).strip("_")

def savefig(name):
    p = FIG_DIR / f"{name}.png"
    plt.savefig(p, dpi=300, bbox_inches="tight")
    print("Saved:", p)

CATEGORY_ORDER = [
    "Lexical baselines",
    "Neural baselines",
    "Transformer baselines",
    "BERT-IDF hybrids",
    "Proposed model",
]

CATEGORY_MARKERS = {
    "Lexical baselines": ["o", "s", "^", "D", "v"],
    "Neural baselines": ["P", "X", "h", "*", ">"],
    "Transformer baselines": ["D", "v", "<", "p", "8"],
    "BERT-IDF hybrids": ["o", "s", "^", "D", "v", "P", "X"],
    "Proposed model": ["*"],
}

def model_category(result):
    family = str(result.get("family", ""))
    model = str(result.get("model", ""))

    if model == CONFIG["proposed_model_name"] or "Proposed" in family:
        return "Proposed model"
    if "Lexical" in family or "TF-IDF" in model or model == "MultinomialNB":
        return "Lexical baselines"
    if "Transformer Context" in family or model in ["BERTBASE", "BERTLARGE", "RoBERTaBASE", "DeBERTaBASE"]:
        return "Transformer baselines"
    if (
        "BERT–IDF Hybrid" in family
        or "BERT-IDF Hybrid" in family
        or model.startswith("BERT +")
        or "BERT–IDF" in model
        or "BERT-IDF" in model
    ):
        return "BERT-IDF hybrids"
    if "Neural" in family or model in ["CNN", "LSTM", "BiLSTM"]:
        return "Neural baselines"
    return "Other"

def bundle_for_dataset(dataset):
    return DATASETS[dataset]

def label_names_for_dataset(dataset):
    return list(bundle_for_dataset(dataset)["label_order"])

def normalized_confusion(y_true, y_pred, n_classes):
    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=np.arange(n_classes),
        normalize="true",
    )
    return np.nan_to_num(cm)

def wrap_label(label, width=11):
    label = str(label).replace("_", " ").replace("-", "- ")
    parts = textwrap.wrap(label, width=width, break_long_words=False)
    return "\n".join(parts) if parts else str(label)

def confusion_label_style(dataset, labels, proposed=False):
    tick_idx = np.arange(len(labels))

    if dataset == "FairCVdb":
        x_labels = [wrap_label(x, width=18) for x in labels]
        y_labels = [wrap_label(x, width=18) for x in labels]
        return tick_idx, x_labels, y_labels, 10, 10, 0

    if dataset == "Resume":
        x_labels = [wrap_label(x, width=10) for x in labels]
        y_labels = [wrap_label(x, width=13) for x in labels]

        if proposed:
            return tick_idx, x_labels, y_labels, 6.5, 6.8, 90

        return tick_idx, x_labels, y_labels, 5.0, 5.4, 90

    x_labels = [wrap_label(x, width=12) for x in labels]
    y_labels = [wrap_label(x, width=12) for x in labels]
    return tick_idx, x_labels, y_labels, 7, 7, 90

def annotate_heatmap(ax, cm):
    n = cm.shape[0]

    if n > 10:
        return

    for i in range(n):
        for j in range(n):
            value = cm[i, j]
            color = "white" if value >= 0.55 else "black"
            ax.text(
                j,
                i,
                f"{value:.2f}",
                ha="center",
                va="center",
                fontsize=8 if n <= 4 else 6,
                color=color,
            )

def confusion_group_layout(dataset, n_models):
    if dataset == "FairCVdb":
        ncols = 2 if n_models > 1 else 1
        nrows = int(np.ceil(n_models / ncols))
        fig_width = 12.0
        fig_height = 4.9 * nrows
        return nrows, ncols, fig_width, fig_height

    if dataset == "Resume":
        ncols = 2 if n_models > 1 else 1
        nrows = int(np.ceil(n_models / ncols))
        fig_width = 16.5
        fig_height = 6.9 * nrows
        return nrows, ncols, fig_width, fig_height

    ncols = 2 if n_models > 1 else 1
    nrows = int(np.ceil(n_models / ncols))
    fig_width = 12.0
    fig_height = 5.0 * nrows
    return nrows, ncols, fig_width, fig_height

def add_subplot_colorbar(fig, ax, im):
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="4%", pad=0.12)
    cbar = fig.colorbar(
        im,
        cax=cax,
        ticks=np.linspace(0.0, 1.0, 6),
    )
    cbar.set_label(
        "Normalized value",
        fontsize=8,
        rotation=90,
        labelpad=10,
    )
    cbar.ax.tick_params(labelsize=7)
    cbar.outline.set_linewidth(0.8)
    return cbar

def plot_metric_bars(metric="Accuracy (%)"):
    if T03_compact_cross_dataset_benchmark.empty:
        return

    for dataset, frame in T03_compact_cross_dataset_benchmark.groupby("Dataset"):
        frame = frame.sort_values(metric, ascending=True)
        plt.figure(figsize=(12, max(5, 0.35 * len(frame))))
        y = np.arange(len(frame))
        plt.barh(y, frame[metric].values)
        plt.yticks(y, frame["Model"].values, fontsize=8)
        plt.xlabel(metric)
        plt.title(f"{metric} comparison — {dataset}")
        plt.grid(axis="x", linestyle="--", alpha=0.4)
        savefig(f"bar_{sname(metric)}_{sname(dataset)}")
        plt.show()

def plot_metric_ci_bars(metric_name, metric_fn, ylabel, file_label):
    if not ALL_RESULTS:
        return

    rows = []
    for r in ALL_RESULTS:
        _, lo, hi = bootstrap_metric_ci(r["y_true"], r["y_pred"], metric_fn)
        value = metric_fn(np.asarray(r["y_true"]), np.asarray(r["y_pred"]))
        rows.append({
            "Dataset": r["dataset"],
            "Model": r["model"],
            "Value": value,
            "Low": lo,
            "High": hi,
        })

    df = pd.DataFrame(rows)

    for dataset, frame in df.groupby("Dataset"):
        frame = frame.sort_values("Value", ascending=True).reset_index(drop=True)
        y = np.arange(len(frame))
        values = frame["Value"].values
        lower = values - frame["Low"].values
        upper = frame["High"].values - values
        scale = 100.0 if metric_name == "Accuracy" else 1.0

        plt.figure(figsize=(12, max(5, 0.35 * len(frame))))
        plt.barh(
            y,
            values * scale,
            xerr=np.vstack([lower * scale, upper * scale]),
            capsize=3,
        )
        plt.yticks(y, frame["Model"].values, fontsize=8)
        plt.xlabel(ylabel)
        plt.title(f"{metric_name} with 95% bootstrap confidence intervals — {dataset}")
        plt.grid(axis="x", linestyle="--", alpha=0.4)
        savefig(f"{file_label}_ci_{sname(dataset)}")
        plt.show()

def plot_accuracy_ci_bars():
    plot_metric_ci_bars(
        "Accuracy",
        lambda yt, yp: accuracy_score(yt, yp),
        "Accuracy (%) with 95% bootstrap CI",
        "accuracy",
    )

def plot_macro_f1_ci_bars():
    plot_metric_ci_bars(
        "Macro-F1",
        lambda yt, yp: f1_score(yt, yp, average="macro", zero_division=0),
        "Macro-F1 with 95% bootstrap CI",
        "macro_f1",
    )

def plot_training_grids():
    curve_specs = [
        ("Accuracy", "train_acc", "val_acc"),
        ("Macro-F1", "train_macro_f1", "val_macro_f1"),
    ]

    categories = [
        "Neural baselines",
        "Transformer baselines",
        "BERT-IDF hybrids",
        "Proposed model",
    ]

    for dataset in sorted({r["dataset"] for r in ALL_RESULTS}):
        for category in categories:
            results = [
                r for r in ALL_RESULTS
                if r["dataset"] == dataset
                and model_category(r) == category
                and r.get("history")
            ]

            if not results:
                continue

            category_markers = CATEGORY_MARKERS.get(category, ["o"])
            marker_cycle = itertools.cycle(category_markers)
            model_markers = {r["model"]: next(marker_cycle) for r in results}

            for metric_label, train_key, val_key in curve_specs:
                nrows = len(results)
                fig, axes = plt.subplots(
                    nrows,
                    2,
                    figsize=(12, 3.4 * nrows),
                    squeeze=False,
                )

                for i, result in enumerate(results):
                    hist = pd.DataFrame(result["history"])
                    epochs = hist["epoch"].values
                    mk = model_markers[result["model"]]

                    ax1 = axes[i, 0]
                    ax1.plot(
                        epochs,
                        hist[train_key].values,
                        label="Train",
                        linestyle="-",
                        marker=mk,
                        markersize=4,
                    )
                    ax1.plot(
                        epochs,
                        hist[val_key].values,
                        label="Validation",
                        linestyle="--",
                        marker=mk,
                        markersize=4,
                        markerfacecolor="white",
                    )
                    ax1.set_title(f"{result['model']} — {metric_label}")
                    ax1.set_xlabel("Epoch")
                    ax1.set_ylabel(metric_label)
                    ax1.grid(True, linestyle="--", alpha=0.4)
                    if i == 0:
                        ax1.legend()

                    ax2 = axes[i, 1]
                    ax2.plot(
                        epochs,
                        hist["train_loss"].values,
                        label="Train loss",
                        linestyle="-",
                        marker=mk,
                        markersize=4,
                    )
                    ax2.plot(
                        epochs,
                        hist["val_loss"].values,
                        label="Validation loss",
                        linestyle="--",
                        marker=mk,
                        markersize=4,
                        markerfacecolor="white",
                    )
                    ax2.set_title(f"{result['model']} — Loss")
                    ax2.set_xlabel("Epoch")
                    ax2.set_ylabel("Loss")
                    ax2.grid(True, linestyle="--", alpha=0.4)
                    if i == 0:
                        ax2.legend()

                fig.suptitle(f"{dataset} — {category} — {metric_label} and Loss", fontsize=14)
                fig.tight_layout(rect=[0, 0, 1, 0.98])
                savefig(f"training_{sname(dataset)}_{sname(category)}_{sname(metric_label)}")
                plt.show()

def draw_confusion_subplot(ax, result, dataset, labels, style_info):
    n_classes = len(labels)
    tick_idx, x_tick_labels, y_tick_labels, x_font, y_font, rotation = style_info

    cm = normalized_confusion(result["y_true"], result["y_pred"], n_classes)

    im = ax.imshow(
        cm,
        interpolation="nearest",
        cmap="Blues",
        vmin=0.0,
        vmax=1.0,
        aspect="equal",
    )

    ax.set_title(result["model"], fontsize=10, pad=8)
    ax.set_xlabel("Predicted", fontsize=8, labelpad=6)
    ax.set_ylabel("True", fontsize=8, labelpad=6)

    ax.set_xticks(tick_idx)
    ax.set_xticklabels(
        x_tick_labels,
        rotation=rotation,
        ha="center" if rotation == 90 else "center",
        fontsize=x_font,
    )

    ax.set_yticks(tick_idx)
    ax.set_yticklabels(y_tick_labels, fontsize=y_font)

    ax.tick_params(axis="x", pad=2, length=2)
    ax.tick_params(axis="y", pad=2, length=2)

    annotate_heatmap(ax, cm)

    return im

def plot_confusion_groups():
    group_categories = [
        "Lexical baselines",
        "Neural baselines",
        "Transformer baselines",
        "BERT-IDF hybrids",
    ]

    for dataset in sorted({r["dataset"] for r in ALL_RESULTS}):
        labels = label_names_for_dataset(dataset)
        style_info = confusion_label_style(dataset, labels)

        for category in group_categories:
            results = [
                r for r in ALL_RESULTS
                if r["dataset"] == dataset and model_category(r) == category
            ]

            if not results:
                continue

            n_models = len(results)
            nrows, ncols, fig_width, fig_height = confusion_group_layout(dataset, n_models)

            fig, axes = plt.subplots(
                nrows,
                ncols,
                figsize=(fig_width, fig_height),
                squeeze=False,
            )

            for idx, result in enumerate(results):
                row = idx // ncols
                col = idx % ncols
                ax = axes[row, col]
                im = draw_confusion_subplot(ax, result, dataset, labels, style_info)
                add_subplot_colorbar(fig, ax, im)

            for idx in range(n_models, nrows * ncols):
                axes[idx // ncols, idx % ncols].axis("off")

            fig.suptitle(
                f"{dataset} — {category}",
                fontsize=14,
                y=0.995,
            )

            if dataset == "FairCVdb":
                fig.subplots_adjust(
                    left=0.08,
                    right=0.94,
                    bottom=0.13,
                    top=0.90,
                    wspace=0.52,
                    hspace=0.42,
                )
            elif dataset == "Resume":
                fig.subplots_adjust(
                    left=0.10,
                    right=0.94,
                    bottom=0.11,
                    top=0.94,
                    wspace=0.62,
                    hspace=0.54,
                )
            else:
                fig.subplots_adjust(
                    left=0.10,
                    right=0.94,
                    bottom=0.12,
                    top=0.90,
                    wspace=0.45,
                    hspace=0.40,
                )

            savefig(f"confusion_group_{sname(dataset)}_{sname(category)}")
            plt.show()

def plot_proposed_confusion():
    for dataset in sorted({r["dataset"] for r in ALL_RESULTS}):
        result = next(
            (
                r for r in ALL_RESULTS
                if r["dataset"] == dataset
                and r["model"] == CONFIG["proposed_model_name"]
            ),
            None,
        )

        if result is None:
            continue

        labels = label_names_for_dataset(dataset)
        style_info = confusion_label_style(dataset, labels, proposed=True)

        if dataset == "FairCVdb":
            fig_size = (7.4, 6.5)
            adjust = dict(left=0.15, right=0.88, bottom=0.16, top=0.88)
        elif dataset == "Resume":
            fig_size = (11.0, 9.8)
            adjust = dict(left=0.23, right=0.88, bottom=0.25, top=0.90)
        else:
            fig_size = (9.5, 8.5)
            adjust = dict(left=0.18, right=0.88, bottom=0.20, top=0.90)

        fig, ax = plt.subplots(figsize=fig_size)

        im = draw_confusion_subplot(ax, result, dataset, labels, style_info)
        ax.set_title(f"{CONFIG['proposed_model_name']} — {dataset}", fontsize=13, pad=12)

        fig.subplots_adjust(**adjust)
        add_subplot_colorbar(fig, ax, im)

        savefig(f"confusion_proposed_{sname(dataset)}")
        plt.show()

def macro_ovr_roc(y_true, probas, n_classes):
    y_true = np.asarray(y_true)
    probas = np.asarray(probas)

    if n_classes == 2 and probas.shape[1] == 2:
        fpr, tpr, _ = roc_curve(y_true, probas[:, 1])
        return fpr, tpr, auc(fpr, tpr)

    y_bin = np.zeros((len(y_true), n_classes), dtype=int)
    y_bin[np.arange(len(y_true)), y_true.astype(int)] = 1

    grid = np.linspace(0.0, 1.0, 500)
    tprs, aucs = [], []

    for c in range(n_classes):
        if len(np.unique(y_bin[:, c])) < 2:
            continue

        fpr, tpr, _ = roc_curve(y_bin[:, c], probas[:, c])
        interp_tpr = np.interp(grid, fpr, tpr)
        interp_tpr[0] = 0.0
        tprs.append(interp_tpr)
        aucs.append(auc(fpr, tpr))

    if not tprs:
        return None, None, np.nan

    mean_tpr = np.mean(np.vstack(tprs), axis=0)
    mean_tpr[-1] = 1.0
    return grid, mean_tpr, float(np.mean(aucs))

def plot_roc_groups():
    group_categories = [
        "Lexical baselines",
        "Neural baselines",
        "Transformer baselines",
        "BERT-IDF hybrids",
    ]

    for dataset in sorted({r["dataset"] for r in ALL_RESULTS}):
        n_classes = len(label_names_for_dataset(dataset))

        for category in group_categories:
            results = [
                r for r in ALL_RESULTS
                if r["dataset"] == dataset and model_category(r) == category
            ]

            if not results:
                continue

            plt.figure(figsize=(7, 6))
            plotted = 0

            for result in results:
                fpr, tpr, roc_auc = macro_ovr_roc(result["y_true"], result["probas"], n_classes)
                if fpr is None:
                    continue
                plt.plot(fpr, tpr, label=f"{result['model']} (AUC={roc_auc:.3f})")
                plotted += 1

            if plotted == 0:
                plt.close()
                continue

            plt.plot([0, 1], [0, 1], linestyle="--")
            plt.xlabel("False Positive Rate")
            plt.ylabel("True Positive Rate")
            plt.title(f"ROC comparison — {dataset} — {category}")
            plt.legend(fontsize=8, loc="lower right")
            plt.grid(True, linestyle="--", alpha=0.4)
            savefig(f"roc_group_{sname(dataset)}_{sname(category)}")
            plt.show()

def plot_proposed_roc():
    for dataset in sorted({r["dataset"] for r in ALL_RESULTS}):
        result = next(
            (
                r for r in ALL_RESULTS
                if r["dataset"] == dataset
                and r["model"] == CONFIG["proposed_model_name"]
            ),
            None,
        )

        if result is None:
            continue

        n_classes = len(label_names_for_dataset(dataset))
        fpr, tpr, roc_auc = macro_ovr_roc(result["y_true"], result["probas"], n_classes)

        if fpr is None:
            continue

        plt.figure(figsize=(7, 6))
        plt.plot(fpr, tpr, label=f"{CONFIG['proposed_model_name']} (AUC={roc_auc:.3f})")
        plt.plot([0, 1], [0, 1], linestyle="--")
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title(f"ROC curve — {CONFIG['proposed_model_name']} — {dataset}")
        plt.legend(loc="lower right")
        plt.grid(True, linestyle="--", alpha=0.4)
        savefig(f"roc_proposed_{sname(dataset)}")
        plt.show()

def plot_efficiency():
    if T03_compact_cross_dataset_benchmark.empty:
        return

    for dataset, frame in T03_compact_cross_dataset_benchmark.groupby("Dataset"):
        frame = frame.dropna(subset=["Inference ms/document", "Accuracy (%)"])
        if frame.empty:
            continue

        plt.figure(figsize=(8, 6))
        plt.scatter(frame["Inference ms/document"], frame["Accuracy (%)"])

        for _, row in frame.iterrows():
            plt.annotate(
                row["Model"],
                (row["Inference ms/document"], row["Accuracy (%)"]),
                fontsize=7,
            )

        plt.xlabel("Inference time (ms/document)")
        plt.ylabel("Accuracy (%)")
        plt.title(f"Accuracy–efficiency Pareto — {dataset}")
        plt.grid(True, linestyle="--", alpha=0.4)
        savefig(f"efficiency_{sname(dataset)}")
        plt.show()

plot_metric_bars("Accuracy (%)")
plot_metric_bars("Macro-F1")
plot_accuracy_ci_bars()
plot_macro_f1_ci_bars()
plot_training_grids()
plot_confusion_groups()
plot_proposed_confusion()
plot_roc_groups()
plot_proposed_roc()
plot_efficiency()

## 11. FairCVdb fairness audit when demographic attributes are aligned


In [ ]:
def fairness_audit():
    rows = []
    res = [r for r in ALL_RESULTS if r["dataset"] == "FairCVdb" and r["model"] == CONFIG["proposed_model_name"]]
    if not res or "FairCVdb" not in DATASETS:
        return pd.DataFrame()

    r = res[0]
    test = DATASETS["FairCVdb"]["test"].reset_index(drop=True)

    for attr in ["gender", "ethnicity"]:
        if attr not in test.columns:
            continue
        vals = test[attr].astype(str).fillna("UNKNOWN").values
        for g in sorted(set(vals)):
            idx = np.where(vals == g)[0]
            if len(idx) < 5:
                continue
            rows.append({
                "Dataset": "FairCVdb",
                "Sensitive Attribute": attr,
                "Group": g,
                "Support": len(idx),
                "Accuracy": accuracy_score(r["y_true"][idx], r["y_pred"][idx]),
                "Macro-F1": f1_score(r["y_true"][idx], r["y_pred"][idx], average="macro"),
            })
    return pd.DataFrame(rows)

T16_fairness_audit = fairness_audit()
ARTICLE_TABLES["T16_fairness_audit"] = T16_fairness_audit

display(
    T16_fairness_audit
    if not T16_fairness_audit.empty
    else pd.DataFrame({"Message": ["No aligned FairCVdb demographic columns available or group support < 5."]})
)


## Model and benchmark summary

**BERTidRAAF (Ours)** stands for **BERT with IDF-aware Rare-Aware Adaptive Fusion**.

The benchmark sequence is:

1. Lexical baselines: TF-IDF + SVM, Logistic Regression, and Naive Bayes.
2. Neural baselines: CNN, LSTM, and BiLSTM.
3. Transformer baselines: BERTBASE, BERTLARGE, RoBERTaBASE, and DeBERTaBASE.
4. BERT–IDF hybrids: BERT+IDF, BERT+CNN, BERT–IDF+CNN, BERT–IDF+LSTM, and BERT–IDF+BiLSTM.
5. Proposed model: BERTidRAAF (Ours).

The reported test performance follows a fixed-seed protocol and includes bootstrap confidence intervals for accuracy and macro-F1. The final BERT–IDF strategy is selected on the validation split and evaluated once on the held-out test split.
